# Part 4 — Advanced SQL & Data Modeling

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 46: Orchestration with Apache Airflow — Your First DAG](#chapter_46_orchestration_with_apache_airflow_your_first_dag)
- [Chapter 47: Building a Real Airflow DAG — watch_events_dag.py](#chapter_47_building_a_real_airflow_dag_watch_events_dag_py)
- [Chapter 47a: Prefect — Lightweight Idempotent Orchestration](#chapter_47a_prefect_lightweight_idempotent_orchestration)
- [Chapter 48: Data Modeling and the Star Schema](#chapter_48_data_modeling_and_the_star_schema)
- [Chapter 49: dbt — Transformations as Code](#chapter_49_dbt_transformations_as_code)
- [Chapter 50: Cloud Data Warehouses — BigQuery, Snowflake, Redshift](#chapter_50_cloud_data_warehouses_bigquery_snowflake_redshift)
- [Chapter 51: Data Governance, Lineage, and the Analytics Engineer Role](#chapter_51_data_governance_lineage_and_the_analytics_engineer_role)
- [Chapter 51a: Dataset Cards, Model Cards & AI Documentation](#chapter_51a_dataset_cards_model_cards_ai_documentation)

---

# Chapter 46: Orchestration with Apache Airflow — Your First DAG

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    SCH[Scheduler\nreads DAG files] --> DR[DAG Run triggered]
    DR --> T1[extract\nPythonOperator]
    T1 -->|XCom: staging_path| T2[validate\nPythonOperator]
    T2 --> T3[load\nPythonOperator]
    T3 --> T4[notify\nPythonOperator]
    T2 -->|failure| R[Retry x2\nthen Alert email]
    T3 -->|success| LOG[Airflow metadata\nDB logs run]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
# skip
# Note: These code blocks show the Airflow DAG structure.
# To run Airflow locally: pip install apache-airflow
# Then: airflow standalone (starts webserver, scheduler, and database)
# DAG files go in ~/airflow/dags/ (or the configured dags_folder)

# ─── DAG Anatomy ────────────────────────────────────────────────────────────

# Standard imports for every DAG file
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from datetime import datetime, timedelta

# Default arguments — applied to all tasks unless overridden
DEFAULT_ARGS = {
    "owner":            "data-team",
    "depends_on_past":  False,          # task doesn't need yesterday's run to succeed
    "email":            ["priya@cinemastream.com"],
    "email_on_failure": True,           # send email when task fails
    "email_on_retry":   False,          # don't spam on retries
    "retries":          2,              # retry failed tasks twice
    "retry_delay":      timedelta(minutes=5),
}

print("DAG default_args defined.")
print("Key fields:")
for k, v in DEFAULT_ARGS.items():
    print(f"  {k}: {v}")

In [ ]:
# ─── Task Dependencies ───────────────────────────────────────────────────────

# In Airflow, >> means "depends on"
# These show the dependency patterns you'll use:

dependency_examples = """
# Sequential pipeline: A → B → C
extract >> validate >> load

# Fan-out: A runs B and C in parallel
extract >> [validate_users, validate_events]

# Fan-in: B and C must both complete before D
[validate_users, validate_events] >> load

# Diamond: extract → [validate_a, validate_b] → load → notify
extract >> validate_a >> load
extract >> validate_b >> load
load >> notify

# The above is equivalent to:
extract >> [validate_a, validate_b] >> load >> notify
"""

print("Dependency syntax examples:")
print(dependency_examples)

In [ ]:
# ─── PythonOperator and XCom ─────────────────────────────────────────────────

# XCom lets tasks share small values (< 48KB)
# Use it for: counts, statuses, file paths, metric values
# Do NOT use it for: DataFrames, large files (put those in S3/GCS)

xcom_pattern = """
def extract(**context):
    # ... extract logic ...
    row_count = 381  # extracted rows
    
    # Push value to XCom
    context['ti'].xcom_push(key='row_count', value=row_count)

def validate(**context):
    # Pull value from upstream task via XCom
    ti = context['ti']
    row_count = ti.xcom_pull(task_ids='extract_task', key='row_count')
    
    if row_count < 100:
        raise ValueError(f"Expected 100+ rows, got {row_count}")
    
    print(f"Validation passed: {row_count} rows")
"""

print("XCom pattern (task communication):")
print(xcom_pattern)

In [ ]:
# ─── Airflow DAG: CinemaStream Watch Events ──────────────────────────────────
# This is the complete DAG that you'll save to cinemastream/pipelines/watch_events_dag.py

WATCH_EVENTS_DAG = '''
"""
CinemaStream Watch Events Ingestion DAG
Chapter 46 — Orchestration with Apache Airflow

Schedule: Daily at 02:00 UTC
Logical date: yesterday (processes previous day's events)
"""

from __future__ import annotations
import logging
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from airflow import DAG
from airflow.operators.python import PythonOperator

logger = logging.getLogger(__name__)

DEFAULT_ARGS = {
    "owner":            "data-team",
    "depends_on_past":  False,
    "email":            ["priya@cinemastream.com"],
    "email_on_failure": True,
    "email_on_retry":   False,
    "retries":          2,
    "retry_delay":      timedelta(minutes=5),
}

# ── Task Functions ──────────────────────────────────────────────────────────

def extract_watch_events(**context) -> None:
    """Extract raw watch events from source to staging area."""
    logical_date = context["logical_date"]
    date_str     = logical_date.strftime("%Y-%m-%d")
    
    logger.info("Extracting watch_events for date=%s", date_str)
    
    # In production: query PostgreSQL read replica for yesterday's events
    # source_conn = PostgresHook("cinemastream_prod_replica").get_conn()
    # df = pd.read_sql(f"SELECT * FROM watch_events WHERE DATE(watch_started) = \\'{date_str}\\'", source_conn)
    
    # For demo: read from CSV
    df = pd.read_csv("cinemastream/data/watch_events.csv", encoding="utf-8")
    df = df[df["watch_started"].str.startswith(date_str)]
    
    # Write to staging file (in production: write to S3/GCS)
    import tempfile, os
    staging_path = os.path.join(tempfile.gettempdir(), f"watch_events_{date_str}.parquet")
    df.to_parquet(staging_path, index=False)
    
    logger.info("Extracted %d rows to %s", len(df), staging_path)
    context["ti"].xcom_push(key="staging_path", value=staging_path)
    context["ti"].xcom_push(key="row_count",    value=len(df))


def validate_watch_events(**context) -> None:
    """Run data quality expectations on the extracted file."""
    ti           = context["ti"]
    staging_path = ti.xcom_pull(task_ids="extract", key="staging_path")
    row_count    = ti.xcom_pull(task_ids="extract", key="row_count")
    
    logger.info("Validating %d rows from %s", row_count, staging_path)
    
    if row_count == 0:
        logger.warning("Zero rows extracted — skipping validation (empty partition)")
        ti.xcom_push(key="validation_passed", value=True)
        return
    
    df = pd.read_parquet(staging_path)
    
    # Expectations
    errors = []
    if df["event_id"].isnull().any():
        errors.append("event_id has NULL values")
    if df["watch_minutes"].lt(0).any():
        errors.append("Negative watch_minutes found")
    if not df["device"].isin(["Mobile", "TV", "Tablet", "Web"]).all():
        errors.append("Unknown device values found")
    
    if errors:
        raise ValueError(f"Validation failed: {errors}")
    
    logger.info("Validation passed for %d rows", len(df))
    ti.xcom_push(key="validation_passed", value=True)


def load_watch_events(**context) -> None:
    """Load validated events to the analytics warehouse."""
    ti           = context["ti"]
    staging_path = ti.xcom_pull(task_ids="extract", key="staging_path")
    row_count    = ti.xcom_pull(task_ids="extract", key="row_count")
    
    if row_count == 0:
        logger.info("Empty partition — nothing to load")
        return
    
    df = pd.read_parquet(staging_path)
    
    # In production: BigQuery load
    # from google.cloud import bigquery
    # bq = bigquery.Client()
    # bq.load_table_from_dataframe(df, "cinemastream.analytics.watch_events")
    
    logger.info("Loaded %d rows to analytics warehouse", len(df))
    ti.xcom_push(key="loaded_rows", value=len(df))


def send_completion_notification(**context) -> None:
    """Send a Slack or email notification on successful run."""
    ti          = context["ti"]
    loaded_rows = ti.xcom_pull(task_ids="load", key="loaded_rows") or 0
    date_str    = context["logical_date"].strftime("%Y-%m-%d")
    
    message = f"[watch_events] {date_str} complete: {loaded_rows} rows loaded"
    logger.info(message)
    # In production: SlackWebhookOperator or EmailOperator


# ── DAG Definition ──────────────────────────────────────────────────────────

with DAG(
    dag_id             = "watch_events_ingestion",
    description        = "Daily ingestion of CinemaStream watch events",
    default_args       = DEFAULT_ARGS,
    schedule_interval  = "0 2 * * *",      # 02:00 UTC daily  (Airflow 2.4+: use schedule= instead)
    start_date         = datetime(2024, 1, 1),
    catchup            = False,            # don't backfill missing runs
    max_active_runs    = 1,                # don't overlap with yesterday's run
    tags               = ["cinemastream", "ingestion", "watch_events"],
) as dag:

    extract = PythonOperator(
        task_id         = "extract",
        python_callable = extract_watch_events,
    )

    validate = PythonOperator(
        task_id         = "validate",
        python_callable = validate_watch_events,
    )

    load = PythonOperator(
        task_id         = "load",
        python_callable = load_watch_events,
    )

    notify = PythonOperator(
        task_id         = "notify",
        python_callable = send_completion_notification,
    )

    # DAG structure: extract → validate → load → notify
    extract >> validate >> load >> notify
'''

print("Full DAG code (save this to cinemastream/pipelines/watch_events_dag.py):")
print(WATCH_EVENTS_DAG[:500], "...")

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("cinemastream/data")
users_df  = pd.read_csv(DATA_DIR / "users.csv",        encoding="utf-8")
events_df = pd.read_csv(DATA_DIR / "watch_events.csv", encoding="utf-8")

# Before Airflow: cron script (what Priya had)
print("=== Before Airflow: Manual cron job ===")
print("""
# /etc/crontab
0 2 * * 1 /usr/bin/python3 /home/priya/weekly_report.py

Problems:
- Fails silently: nobody knows until Rohan asks
- No retry: if the DB is down at 2am, the report just doesn't run
- No visibility: no log of what ran, when, with what result
- Hard to debug: stdout goes to /dev/null
- No dependency management: if upstream data isn't ready, runs anyway
""")

# After Airflow: the same pipeline with observability + resilience
print("=== After Airflow: Monitored, retried, observable ===")
airflow_improvements = {
    "Visibility":    "Airflow UI shows every task run, status, duration, and log",
    "Alerting":      "email_on_failure=True → Priya gets email within 5 minutes of failure",
    "Retry":         "retries=2 → if the DB is flaky, the task retries before alerting",
    "Dependencies":  "validate >> load → load won't run if validate fails",
    "Concurrency":   "max_active_runs=1 → last week's run can't overlap with this week's",
    "Backfill":      "catchup=False → won't try to run missed dates from start_date",
    "History":       "Every run is logged — you can answer 'what ran on March 3?'",
}
for k, v in airflow_improvements.items():
    print(f"  {k}: {v}")

In [ ]:
# Simulating Airflow task execution without Airflow installed
# (demonstrates the same logic that runs inside Airflow tasks)

class MockTaskInstance:
    """Minimal XCom simulator for demonstration."""
    def __init__(self):
        self._xcoms = {}
    def xcom_push(self, key, value):
        self._xcoms[key] = value
    def xcom_pull(self, task_ids, key):
        return self._xcoms.get(key)

def demo_dag_run():
    """Simulate one complete DAG run."""
    ti = MockTaskInstance()
    context = {"ti": ti, "logical_date": datetime(2023, 1, 15)}
    
    print("DAG Run: watch_events_ingestion — 2023-01-15")
    print()
    
    # Task 1: Extract
    print("[extract] Starting...")
    date_str = "2023-01-15"
    batch = events_df[events_df["watch_started"].str.startswith(date_str)].copy()
    ti.xcom_push(key="row_count", value=len(batch))
    import tempfile, os
    ti.xcom_push(key="staging_path", value=os.path.join(tempfile.gettempdir(), f"we_{date_str}.parquet"))
    print(f"[extract] Done: {len(batch)} rows extracted for {date_str}")
    
    # Task 2: Validate
    print("\n[validate] Starting...")
    row_count = ti.xcom_pull(task_ids="extract", key="row_count")
    if row_count == 0:
        print(f"[validate] Empty partition — OK, skipping")
    else:
        errors = []
        if batch["event_id"].isnull().any():
            errors.append("event_id NULLs")
        if batch["watch_minutes"].lt(0).any():
            errors.append("negative watch_minutes")
        if errors:
            raise ValueError(f"Validation failed: {errors}")
        print(f"[validate] Done: {row_count} rows passed all expectations")
    
    # Task 3: Load
    print("\n[load] Starting...")
    print(f"[load] Done: {row_count} rows written to analytics warehouse")
    ti.xcom_push(key="loaded_rows", value=row_count)
    
    # Task 4: Notify
    print("\n[notify] Sending completion notification...")
    loaded = ti.xcom_pull(task_ids="load", key="loaded_rows") or 0
    print(f"[notify] → 'watch_events 2023-01-15: {loaded} rows loaded successfully'")

demo_dag_run()

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
dependency_chain = """
extract_users >> validate_users
extract_events >> validate_events
[validate_users, validate_events] >> load_to_warehouse >> send_notification

Visual:
extract_users ──→ validate_users ──┐
                                    ├──→ load_to_warehouse ──→ send_notification
extract_events → validate_events ──┘
"""

print(dependency_chain)

In [ ]:
dag_design = {
    "dag_id":    "weekly_metrics_email",
    "schedule":  "0 2 * * 1",   # 02:00 UTC every Monday
    "start_date": "2024-01-01",
    "catchup":   False,
    "max_active_runs": 1,
    "tasks": [
        {
            "task_id":  "extract_metrics",
            "operator": "PythonOperator",
            "does":     "Runs the weekly_metrics SQL query, pushes metrics dict via XCom"
        },
        {
            "task_id":  "validate_metrics",
            "operator": "PythonOperator",
            "does":     "Pulls metrics from XCom, runs validation suite, raises ValueError if failed"
        },
        {
            "task_id":  "format_email",
            "operator": "PythonOperator",
            "does":     "Pulls metrics from XCom, renders email body from Jinja template"
        },
        {
            "task_id":  "send_email",
            "operator": "EmailOperator",
            "does":     "Sends formatted email to Rohan, Dharani, and distribution list"
        },
        {
            "task_id":  "log_run",
            "operator": "PythonOperator",
            "does":     "Writes run metadata (timestamp, metrics values) to metrics_pipeline_runs table"
        },
    ],
    "dependencies": "extract_metrics >> validate_metrics >> format_email >> send_email >> log_run",
}

print("=== Weekly Metrics DAG Design ===")
print(f"Schedule: {dag_design['schedule']} (2am UTC every Monday)")
print(f"Dependencies: {dag_design['dependencies']}")
print("\nTasks:")
for t in dag_design["tasks"]:
    print(f"  {t['task_id']} ({t['operator']}): {t['does']}")

---

# Chapter 47: Building a Real Airflow DAG — watch_events_dag.py

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    LIB[Library functions\npure Python, testable] --> TF[Task functions\ncall library functions]
    TF --> DAG[DAG definition\noperators + dependencies]
    DAG -->|happy path| EP[Empty partition?\nskip gracefully]
    DAG -->|data present| VAL[Validate\nrun_watch_event_validation]
    VAL -->|errors| FAIL[Raise ValueError\nAirflow marks Failed]
    VAL -->|clean| LOAD[idempotent_load\nDELETE partition → INSERT]
    LOAD --> LOG[XCom: loaded_rows\nstructured log]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas

In [ ]:
# Testing Airflow DAG logic without Airflow installed
# Key insight: task functions are just Python functions. Test them directly.

import pandas as pd
import sqlite3
import tempfile
import os
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("cinemastream/data")

### Testing Task Functions Directly

In [ ]:
# Mock the context object that Airflow passes to tasks
class MockXCom:
    """Minimal XCom simulation for unit testing."""
    def __init__(self):
        self._store: dict = {}
    
    def xcom_push(self, key: str, value) -> None:
        self._store[key] = value
    
    def xcom_pull(self, task_ids: str, key: str):
        return self._store.get(key)

class MockContext:
    def __init__(self, date: str = "2023-01-15"):
        self.ti           = MockXCom()
        self.logical_date = datetime.fromisoformat(date)
    
    def __getitem__(self, key):
        return getattr(self, key)

# Test the extract function logic
def test_extract_logic():
    events_df = pd.read_csv(DATA_DIR / "watch_events.csv", encoding="utf-8")
    
    date_str = "2023-01-15"
    batch    = events_df[events_df["watch_started"].str.startswith(date_str)].copy()
    
    # Simulate extract: write to temp staging file
    with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as f:
        staging_path = f.name
    
    batch.to_parquet(staging_path, index=False)
    
    # Verify
    loaded = pd.read_parquet(staging_path)
    assert len(loaded) == len(batch), f"Expected {len(batch)} rows, got {len(loaded)}"
    assert loaded["event_id"].notna().all(), "event_id should not have NULLs"
    
    os.unlink(staging_path)
    print(f"test_extract_logic PASSED: {len(batch)} rows for {date_str}")
    return len(batch)

n_rows = test_extract_logic()

### Testing the Validation Logic

In [ ]:
from typing import Optional

def run_watch_event_validation(df: pd.DataFrame) -> list[str]:
    """
    Core validation logic, extracted from the Airflow task for testability.
    Returns list of error strings. Empty = passed.
    """
    VALID_DEVICES   = {"Mobile", "TV", "Tablet", "Web"}
    VALID_COUNTRIES = {"SG", "MY", "ID", "PH", "TH", "VN", "IN"}
    
    errors = []
    
    if df["event_id"].isnull().any():
        errors.append(f"event_id: {df['event_id'].isnull().sum()} NULL values")
    if df["event_id"].duplicated().any():
        errors.append(f"event_id: {df['event_id'].duplicated().sum()} duplicates")
    if df["user_id"].isnull().any():
        errors.append(f"user_id: {df['user_id'].isnull().sum()} NULL values")
    
    bad_mins = df[~df["watch_minutes"].between(1, 300)]
    if len(bad_mins) > 0:
        errors.append(f"watch_minutes out of [1,300]: {len(bad_mins)} rows")
    
    bad_devices = df[~df["device"].isin(VALID_DEVICES)]["device"].unique().tolist()
    if bad_devices:
        errors.append(f"Unknown devices: {bad_devices}")
    
    bad_countries = df[~df["country"].isin(VALID_COUNTRIES)]["country"].unique().tolist()
    if bad_countries:
        errors.append(f"Unknown countries: {bad_countries}")
    
    return errors


# Test 1: clean data
clean_df = pd.read_csv(DATA_DIR / "watch_events.csv", encoding="utf-8")
errors   = run_watch_event_validation(clean_df)
print(f"Clean data: {len(errors)} errors (expected 0)")
assert len(errors) == 0

# Test 2: dirty data — inject failures
dirty_df = clean_df.copy()
dirty_df.loc[0, "event_id"]     = None        # NULL event_id
dirty_df.loc[1, "watch_minutes"] = -5          # negative minutes
dirty_df.loc[2, "device"]        = "VR"        # unknown device
dirty_df.loc[3, "country"]       = "JP"        # outside our market

errors_dirty = run_watch_event_validation(dirty_df)
print(f"\nDirty data errors ({len(errors_dirty)}):")
for e in errors_dirty:
    print(f"  - {e}")

### Testing the Full Pipeline End-to-End

In [ ]:
import numpy as np

def run_pipeline_end_to_end(date_str: str = "2023-01-15") -> dict:
    """
    Simulate one complete DAG run without Airflow.
    Returns final stats dict.
    """
    events_df = pd.read_csv(DATA_DIR / "watch_events.csv", encoding="utf-8")
    
    # Task 1: Extract
    batch = events_df[events_df["watch_started"].str.startswith(date_str)].copy()
    row_count = len(batch)
    
    # Task 2: Validate
    if row_count > 0:
        errors = run_watch_event_validation(batch)
        if errors:
            raise ValueError(f"Validation failed: {errors}")
    
    # Task 3: Load (transform + load)
    if row_count > 0:
        conditions  = [batch["watch_minutes"] < 30, batch["watch_minutes"].between(30, 89)]
        batch["session_bucket"] = np.select(conditions, ["short", "medium"], default="long")
        batch["loaded_at"]      = datetime.utcnow().isoformat()
        
        conn = sqlite3.connect(":memory:")
        batch.to_sql("watch_events_daily", conn, if_exists="replace", index=False)
        loaded_count = pd.read_sql("SELECT COUNT(*) AS n FROM watch_events_daily", conn)['n'][0]
        conn.close()
    else:
        loaded_count = 0
    
    return {
        "date":          date_str,
        "extracted":     row_count,
        "validated":     row_count,
        "loaded":        loaded_count,
        "status":        "success",
    }


# Run for multiple dates
for date in ["2023-01-15", "2023-02-04", "2023-06-20"]:
    result = run_pipeline_end_to_end(date)
    status = "SUCCESS" if result["loaded"] == result["extracted"] else "PARTIAL"
    print(f"[{status}] {result['date']}: extracted={result['extracted']}, loaded={result['loaded']}")

### Edge Case: Empty Partition

In [ ]:
def test_empty_partition():
    """Pipeline must handle dates with no data gracefully."""
    result = run_pipeline_end_to_end(date_str="2024-12-31")  # future date with no data
    assert result["extracted"] == 0
    assert result["loaded"]    == 0
    assert result["status"]    == "success"
    print(f"Empty partition test PASSED: {result}")

test_empty_partition()

### Backfill Simulation

In [ ]:
def run_backfill(start_date: str, end_date: str) -> list[dict]:
    """
    Simulate running the pipeline for a historical date range.
    In Airflow: `airflow dags backfill -s 2023-01-01 -e 2023-01-31 watch_events_ingestion`
    """
    from datetime import date, timedelta
    
    current = datetime.strptime(start_date, "%Y-%m-%d").date()
    end     = datetime.strptime(end_date,   "%Y-%m-%d").date()
    results = []
    
    while current <= end:
        result = run_pipeline_end_to_end(current.strftime("%Y-%m-%d"))
        results.append(result)
        current += timedelta(days=1)
    
    return results

backfill_results = run_backfill("2023-01-01", "2023-01-07")
total_rows = sum(r["loaded"] for r in backfill_results)
non_empty  = sum(1 for r in backfill_results if r["loaded"] > 0)
print(f"Backfill 2023-01-01 to 2023-01-07:")
print(f"  {len(backfill_results)} days processed | {non_empty} non-empty | {total_rows} total rows loaded")
for r in backfill_results:
    print(f"  {r['date']}: {r['loaded']} rows")

## 3. CinemaStream in Practice

In [ ]:
# Code review simulation — common review comments for Airflow DAGs

code_review_items = [
    {
        "line":    "from pathlib import Path",
        "status":  "APPROVE",
        "comment": "Good — using Path for portable file handling.",
    },
    {
        "line":    "staging_path = os.path.join(tempfile.gettempdir(), f'watch_events_{date_str}.parquet')",
        "status":  "COMMENT",
        "comment": "Use Path objects and ensure the staging dir is created at DAG level, not in task. "
                   "If two workers run simultaneously, there could be a race on mkdir.",
    },
    {
        "line":    "context['ti'].xcom_push(key='row_count', value=len(df))",
        "status":  "APPROVE",
        "comment": "Good — always push row_count, even for empty batches. "
                   "Downstream tasks can use this to skip work cleanly.",
    },
    {
        "line":    "if row_count == 0:\n    logger.warning('Empty partition — skipping validation')",
        "status":  "APPROVE",
        "comment": "Correct handling of empty partitions. Empty = OK for new markets or "
                   "historical backfills. Don't fail on empty.",
    },
    {
        "line":    "df.to_sql('watch_events_daily', conn, if_exists='append')",
        "status":  "BLOCK",
        "comment": "BLOCK: append-only load is not idempotent. If the pipeline re-runs for the same "
                   "date, you'll insert duplicates. Use if_exists='replace' + date partition, or "
                   "upsert on event_id primary key.",
    },
    {
        "line":    "max_active_runs=1",
        "status":  "APPROVE",
        "comment": "Essential. Prevents overlap when backfilling and the previous day's run "
                   "is still running.",
    },
]

print("=== Code Review: watch_events_dag.py ===")
for item in code_review_items:
    status_emoji = {"APPROVE": "✓", "COMMENT": "→", "BLOCK": "✗"}[item["status"]]
    print(f"\n{status_emoji} [{item['status']}]")
    print(f"  Code: {item['line'][:60]}...")
    print(f"  Comment: {item['comment']}")

In [ ]:
# Fix the BLOCK comment: idempotent load with date partition
def idempotent_load(df: pd.DataFrame, conn, logical_date: str) -> int:
    """
    Load a day's events idempotently.
    Deletes the existing partition for this date, then inserts fresh.
    Running twice produces the same result.
    """
    # Create table if not exists
    conn.execute("""
    CREATE TABLE IF NOT EXISTS watch_events_daily (
        event_id      INTEGER,
        user_id       INTEGER,
        movie_id      INTEGER,
        watch_started TEXT,
        watch_minutes INTEGER,
        completed     INTEGER,
        device        TEXT,
        country       TEXT,
        session_bucket TEXT,
        loaded_at     TEXT,
        partition_date TEXT,   -- explicit partition column for easy deletion
        PRIMARY KEY (event_id)
    )
    """)
    
    # Delete existing rows for this partition (safe to re-run)
    conn.execute("DELETE FROM watch_events_daily WHERE partition_date = ?", (logical_date,))
    
    # Insert fresh rows
    if len(df) > 0:
        df = df.copy()
        df["partition_date"] = logical_date
        df["loaded_at"]      = datetime.utcnow().isoformat()
        df.to_sql("watch_events_daily", conn, if_exists="append", index=False)
    
    conn.commit()
    count = pd.read_sql(
        "SELECT COUNT(*) AS n FROM watch_events_daily WHERE partition_date = ?",
        conn, params=(logical_date,)
    )['n'][0]
    return int(count)


# Test idempotency
test_conn = sqlite3.connect(":memory:")
clean_events = pd.read_csv(DATA_DIR / "watch_events.csv", encoding="utf-8")
batch = clean_events[clean_events["watch_started"].str.startswith("2023-01-15")].copy()

import numpy as np
conditions = [batch["watch_minutes"] < 30, batch["watch_minutes"].between(30, 89)]
batch["session_bucket"] = np.select(conditions, ["short", "medium"], default="long")

# Run 1
count1 = idempotent_load(batch, test_conn, "2023-01-15")
# Run 2 (same date) — must produce same count
count2 = idempotent_load(batch, test_conn, "2023-01-15")

print(f"Run 1: {count1} rows | Run 2: {count2} rows | Idempotent: {count1 == count2}")
assert count1 == count2, "Idempotency violated!"
print("Idempotency test PASSED.")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
def test_validation_suite():
    events_df = pd.read_csv("cinemastream/data/watch_events.csv", encoding="utf-8")
    
    # (a) Clean data passes
    errors_clean = run_watch_event_validation(events_df)
    assert len(errors_clean) == 0, f"Expected no errors, got: {errors_clean}"
    print("(a) Clean data: PASS")
    
    # (b) Negative watch_minutes fails
    bad_mins_df = events_df.copy()
    bad_mins_df.loc[0, "watch_minutes"] = -10
    errors_mins = run_watch_event_validation(bad_mins_df)
    assert any("watch_minutes" in e for e in errors_mins), f"Expected watch_minutes error, got: {errors_mins}"
    print(f"(b) Negative minutes: PASS (caught: {errors_mins[0]})")
    
    # (c) Unknown device fails
    bad_dev_df = events_df.copy()
    bad_dev_df.loc[0, "device"] = "SmartFridge"
    errors_dev = run_watch_event_validation(bad_dev_df)
    assert any("SmartFridge" in e for e in errors_dev), f"Expected device error, got: {errors_dev}"
    print(f"(c) Unknown device: PASS (caught: {errors_dev[0]})")
    
    # (d) All issues present
    multi_df = events_df.copy()
    multi_df.loc[0, "watch_minutes"] = -5
    multi_df.loc[1, "device"]        = "Hologram"
    multi_df.loc[2, "country"]       = "ZZ"
    errors_multi = run_watch_event_validation(multi_df)
    assert len(errors_multi) >= 3, f"Expected 3+ errors, got {len(errors_multi)}: {errors_multi}"
    print(f"(d) Multi-issue: PASS ({len(errors_multi)} errors detected)")
    
    print("\nAll validation tests PASSED.")

test_validation_suite()

In [ ]:
sensor_config = {
    "task_id":              "wait_for_ingestion",
    "operator":             "ExternalTaskSensor",
    "external_dag_id":      "watch_events_ingestion",
    "external_task_id":     "load",
    "mode":                 "reschedule",  # releases worker slot between polls
    "poke_interval":        60,            # check every 60 seconds
    "timeout":              3600,          # fail if not done within 1 hour
    "allowed_states":       ["success"],
    "execution_delta":      "timedelta(days=0)",  # same logical_date
}

print("ExternalTaskSensor configuration:")
for k, v in sensor_config.items():
    print(f"  {k}: {v}")
print("\nThis task would hold the weekly_metrics_email DAG until")
print("watch_events_ingestion's 'load' task succeeds for the same date.")

In [ ]:
def dag_health_check(sample_dates: list[str] = None) -> dict:
    """Verify pipeline logic is correct on sample dates."""
    if sample_dates is None:
        sample_dates = ["2023-01-15", "2023-02-04", "2024-12-31"]
    
    results = []
    all_passed = True
    
    print("=== DAG Health Check: watch_events_ingestion ===")
    for date_str in sample_dates:
        try:
            result = run_pipeline_end_to_end(date_str)
            passed = result["extracted"] == result["loaded"]
            if not passed:
                all_passed = False
            status = "PASS" if passed else "FAIL"
            print(f"  [{status}] {date_str}: extracted={result['extracted']}, loaded={result['loaded']}")
            results.append({"date": date_str, "status": status, **result})
        except Exception as e:
            all_passed = False
            print(f"  [ERROR] {date_str}: {e}")
            results.append({"date": date_str, "status": "ERROR", "error": str(e)})
    
    summary = {
        "total_dates": len(sample_dates),
        "passed":      sum(1 for r in results if r["status"] == "PASS"),
        "failed":      sum(1 for r in results if r["status"] in ("FAIL", "ERROR")),
        "deploy_safe": all_passed,
    }
    
    print(f"\nSummary: {summary['passed']}/{summary['total_dates']} passed")
    print(f"Deploy safe: {'YES' if summary['deploy_safe'] else 'NO — fix failures before deploying'}")
    return summary

health = dag_health_check()
assert health["deploy_safe"], "Health check failed — do not deploy"

---

# Chapter 47a: Prefect — Lightweight Idempotent Orchestration

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### Flows and tasks

In [ ]:
# skip
# pip install prefect
from prefect import flow, task

@task
def extract(source: str) -> list[dict]:
    """Pull records from a source."""
    print(f"Extracting from {source}...")
    return [{"id": 1, "val": 10}, {"id": 2, "val": 20}]


@task
def transform(records: list[dict]) -> list[dict]:
    """Double every value."""
    return [{"id": r["id"], "val": r["val"] * 2} for r in records]


@task
def load(records: list[dict], dest: str) -> int:
    """Write records to destination. Returns count."""
    print(f"Loading {len(records)} records to {dest}.")
    return len(records)


@flow(name="etl-demo")
def etl_pipeline(source: str = "api", dest: str = "warehouse") -> int:
    raw      = extract(source)
    cleaned  = transform(raw)
    n_loaded = load(cleaned, dest)
    return n_loaded


if __name__ == "__main__":
    result = etl_pipeline()
    print(f"Done. Loaded {result} records.")

### Retries

In [ ]:
# skip
from prefect import flow, task
import random
import json

@task(retries=3, retry_delay_seconds=1)
def flaky_api_call(endpoint: str) -> dict:
    """Simulates an API that fails 60% of the time."""
    if random.random() < 0.60:
        raise ConnectionError(f"Timeout on {endpoint}")
    return {"status": "ok", "data": [1, 2, 3]}


@flow
def fetch_with_retry() -> dict:
    random.seed(42)   # makes the demo deterministic: fails twice, passes third
    return flaky_api_call("https://api.cinemastream.com/events")


if __name__ == "__main__":
    result = fetch_with_retry()
    print(json.dumps(result))

### Idempotency — the same guarantee, different syntax

In [ ]:
# skip
from prefect import flow, task
from datetime import date

# Simulated in-memory "already loaded" store
_already_loaded: set[str] = set()


@task
def load_partition(partition_date: date, records: list[dict]) -> int:
    key = str(partition_date)
    if key in _already_loaded:
        print(f"  Partition {key} already loaded — skipping.")
        return 0
    _already_loaded.add(key)
    print(f"  Loaded {len(records)} records for {key}.")
    return len(records)


@flow
def idempotent_ingestion(run_date: date, records: list[dict]) -> int:
    return load_partition(run_date, records)


if __name__ == "__main__":
    test_records = [{"id": 1}, {"id": 2}]
    target = date(2024, 6, 15)

    n1 = idempotent_ingestion(target, test_records)
    print(f"First run:  loaded {n1}")

    n2 = idempotent_ingestion(target, test_records)
    print(f"Second run: loaded {n2}  (idempotent — no double-write)")

### Prefect vs Airflow — decision table

In [ ]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class OrchestrationContext:
    num_pipelines: int           # how many distinct flows/DAGs
    needs_shared_ui: bool        # team needs a monitoring dashboard
    already_runs_airflow: bool   # Airflow already in the stack
    sub_minute_triggers: bool    # need <60s scheduling or event-driven
    ops_team_available: bool     # dedicated team to maintain infra


def recommend_orchestrator(ctx: OrchestrationContext) -> Literal["Prefect", "Airflow"]:
    if ctx.already_runs_airflow:
        return "Airflow"
    if ctx.num_pipelines > 20 and ctx.needs_shared_ui and ctx.ops_team_available:
        return "Airflow"
    if ctx.sub_minute_triggers:
        return "Prefect"
    if ctx.num_pipelines <= 10:
        return "Prefect"
    return "Airflow"


contexts = [
    ("New startup, 3 pipelines, no ops team",
     OrchestrationContext(3, False, False, False, False)),
    ("Enterprise, 50 DAGs, Airflow already running",
     OrchestrationContext(50, True, True, False, True)),
    ("Event-driven ingestion, fires on webhook",
     OrchestrationContext(5, False, False, True, False)),
    ("Mid-size team, 15 flows, needs dashboard",
     OrchestrationContext(15, True, False, False, True)),
]

for name, ctx in contexts:
    print(f"{recommend_orchestrator(ctx):8s}  {name}")

## 3. CinemaStream in Practice

In [ ]:
# skip
# cinemastream/pipelines/subscription_sync_flow.py
"""
Prefect flow: sync subscription events from billing API to warehouse.
Schedule: every 30 minutes via system cron  →  python subscription_sync_flow.py
Chapter: 047a — Prefect

pip install prefect
"""

from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional
from prefect import flow, task
from prefect.logging import get_run_logger


@dataclass
class SubscriptionEvent:
    event_id: str
    user_id: int
    event_type: str          # UPGRADE | DOWNGRADE | CANCEL | REACTIVATE
    old_plan: Optional[str]
    new_plan: Optional[str]
    occurred_at: datetime


@dataclass
class SyncResult:
    run_date: str
    extracted: int = 0
    loaded: int = 0
    skipped: int = 0
    errors: list[str] = field(default_factory=list)

    @property
    def success(self) -> bool:
        return len(self.errors) == 0


# --- Simulated data store (in production: DuckDB / BigQuery / Postgres) ---
_loaded_event_ids: set[str] = set()


def _simulate_billing_api(limit: int = 10) -> list[SubscriptionEvent]:
    """Return synthetic subscription events (replaces real API call)."""
    now = datetime.now(timezone.utc)
    return [
        SubscriptionEvent("evt_001", 12, "UPGRADE",     "Basic",   "Premium", now),
        SubscriptionEvent("evt_002", 47, "CANCEL",      "Basic",   None,      now),
        SubscriptionEvent("evt_003", 91, "REACTIVATE",  None,      "Free",    now),
        SubscriptionEvent("evt_004", 33, "DOWNGRADE",   "Premium", "Basic",   now),
        SubscriptionEvent("evt_005", 78, "UPGRADE",     "Free",    "Basic",   now),
    ][:limit]


# --- Tasks ---

@task(name="extract-subscription-events", retries=2, retry_delay_seconds=30)
def extract_events(since_minutes: int = 30) -> list[SubscriptionEvent]:
    logger = get_run_logger()
    events = _simulate_billing_api()
    logger.info(f"Extracted {len(events)} events from billing API.")
    return events


@task(name="validate-events")
def validate_events(events: list[SubscriptionEvent]) -> list[SubscriptionEvent]:
    logger = get_run_logger()
    valid = []
    for e in events:
        if not e.event_id or not e.user_id:
            logger.warning(f"Skipping malformed event: {e}")
            continue
        if e.event_type not in {"UPGRADE", "DOWNGRADE", "CANCEL", "REACTIVATE"}:
            logger.warning(f"Unknown event_type '{e.event_type}' for {e.event_id}")
            continue
        valid.append(e)
    logger.info(f"Validated {len(valid)}/{len(events)} events.")
    return valid


@task(name="load-events", retries=1, retry_delay_seconds=10)
def load_events(events: list[SubscriptionEvent]) -> SyncResult:
    logger = get_run_logger()
    result = SyncResult(run_date=datetime.now(timezone.utc).strftime("%Y-%m-%d"))
    result.extracted = len(events)

    for e in events:
        if e.event_id in _loaded_event_ids:
            logger.debug(f"  {e.event_id}: already loaded — skip.")
            result.skipped += 1
            continue
        _loaded_event_ids.add(e.event_id)
        logger.info(f"  {e.event_id}: {e.event_type} user {e.user_id} "
                    f"{e.old_plan or '?'} → {e.new_plan or 'churned'}")
        result.loaded += 1

    return result


# --- Flow ---

@flow(name="subscription-sync", log_prints=True)
def subscription_sync_flow(since_minutes: int = 30) -> SyncResult:
    """Idempotent sync of subscription events from billing API to warehouse."""
    raw     = extract_events(since_minutes)
    valid   = validate_events(raw)
    result  = load_events(valid)

    print(f"Sync complete: {result.loaded} loaded, "
          f"{result.skipped} skipped, {len(result.errors)} errors.")
    return result


if __name__ == "__main__":
    r = subscription_sync_flow()
    print(f"Success: {r.success}")

    # Re-run proves idempotency
    print("\n--- Re-run (idempotency check) ---")
    r2 = subscription_sync_flow()
    print(f"Second run loaded: {r2.loaded} (expect 0)")

```
# crontab entry — run every 30 minutes
*/30 * * * * cd /opt/cinemastream && python pipelines/subscription_sync_flow.py >> /var/log/sub_sync.log 2>&1
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
# skip
from prefect import flow, task

@task(retries=1)
def count_words(strings: list[str]) -> int:
    return sum(len(s.split()) for s in strings)

@flow
def word_count_flow(strings: list[str]) -> int:
    return count_words(strings)

if __name__ == "__main__":
    result = word_count_flow(["hello world", "prefect is fun", "one two three four"])
    print(f"Total words: {result}")

In [ ]:
# skip
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional
from prefect import flow, task

@dataclass
class SubscriptionEvent:
    event_id: str
    user_id: int
    event_type: str
    old_plan: Optional[str]
    new_plan: Optional[str]
    occurred_at: datetime

@dataclass
class SyncResult:
    run_date: str
    extracted: int = 0
    loaded: int = 0
    skipped: int = 0
    loaded_ids: list[str] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)

_loaded: set[str] = set()

@task
def extract_events() -> list[SubscriptionEvent]:
    now = datetime.now(timezone.utc)
    return [
        SubscriptionEvent("evt_001", 12, "UPGRADE",    "Basic",   "Premium", now),
        SubscriptionEvent("evt_002", 47, "CANCEL",     "Basic",   None,      now),
        SubscriptionEvent("evt_003", 91, "REACTIVATE", None,      "Free",    now),
    ]

@task
def load_events(events: list[SubscriptionEvent]) -> SyncResult:
    result = SyncResult(run_date=datetime.now(timezone.utc).strftime("%Y-%m-%d"))
    result.extracted = len(events)
    for e in events:
        if e.event_id in _loaded:
            result.skipped += 1
            continue
        _loaded.add(e.event_id)
        result.loaded += 1
        result.loaded_ids.append(e.event_id)
    return result

@task
def summarise_events(events: list[SubscriptionEvent], result: SyncResult) -> None:
    loaded_set = set(result.loaded_ids)
    counts: dict[str, int] = {}
    for e in events:
        if e.event_id in loaded_set:
            counts[e.event_type] = counts.get(e.event_type, 0) + 1
    summary = ", ".join(f"{k}: {v}" for k, v in sorted(counts.items()))
    print(f"Event summary: {summary}")

@flow(log_prints=True)
def subscription_sync_with_summary() -> None:
    events = extract_events()
    result = load_events(events)
    summarise_events(events, result)
    print(f"Loaded: {result.loaded}, Skipped: {result.skipped}")

if __name__ == "__main__":
    subscription_sync_with_summary()

---

# Chapter 48: Data Modeling and the Star Schema

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
erDiagram
    FACT_WATCH_EVENTS {
        int event_id PK
        int user_key FK
        int movie_key FK
        int date_key FK
        int device_key FK
        float watch_minutes
        boolean completed_flag
        float revenue_attributed_sgd
    }
    DIM_USER {
        int user_key PK
        int user_id
        string country
        string plan
        boolean is_churned
    }
    DIM_MOVIE {
        int movie_key PK
        int movie_id
        string title
        string genre
        int release_year
    }
    DIM_DATE {
        int date_key PK
        date full_date
        int year
        int month
        string month_name
        string day_name
        int is_weekend
    }

    FACT_WATCH_EVENTS }|--|| DIM_USER : "who watched"
    FACT_WATCH_EVENTS }|--|| DIM_MOVIE : "what was watched"
    FACT_WATCH_EVENTS }|--|| DIM_DATE : "when"
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
from datetime import datetime

### Star Schema Design — CinemaStream

In [ ]:
# The CinemaStream star schema design
star_schema_design = {
    "fact_table": {
        "name": "fact_watch_events",
        "grain": "one row per watch event",
        "foreign_keys": ["user_key", "movie_key", "date_key", "device_key"],
        "measures": ["watch_minutes", "completed_flag", "revenue_attributed_sgd"],
        "rationale": "Watch events are the central business fact — every metric derives from them"
    },
    "dimension_tables": [
        {
            "name": "dim_user",
            "grain": "one row per user (current state)",
            "key": "user_key",
            "attributes": ["user_id", "name", "email", "country", "language_pref",
                          "plan", "signup_date", "tenure_days", "is_churned"],
        },
        {
            "name": "dim_movie",
            "grain": "one row per movie",
            "key": "movie_key",
            "attributes": ["movie_id", "title", "genre", "original_lang",
                          "release_year", "runtime_min", "is_original_content"],
        },
        {
            "name": "dim_date",
            "grain": "one row per calendar date (pre-populated for 5 years)",
            "key": "date_key",
            "attributes": ["full_date", "year", "quarter", "month", "week",
                          "day_of_week", "day_name", "is_weekend", "is_holiday"],
        },
        {
            "name": "dim_device",
            "grain": "one row per device type",
            "key": "device_key",
            "attributes": ["device_type", "device_category", "screen_size_bucket"],
        },
    ]
}

print("=== CinemaStream Star Schema ===")
print(f"\nFact Table: {star_schema_design['fact_table']['name']}")
print(f"  Grain: {star_schema_design['fact_table']['grain']}")
print(f"  Measures: {star_schema_design['fact_table']['measures']}")

print("\nDimension Tables:")
for dim in star_schema_design["dimension_tables"]:
    print(f"  {dim['name']} — grain: {dim['grain']}")

### Building the Star Schema in SQLite

In [ ]:
conn = sqlite3.connect(":memory:")

# Dimension: date (pre-populated — a crucial design decision)
# The date dimension is created once and never changes
# It allows: WHERE dim_date.is_weekend = 1 (no date arithmetic in queries)

from datetime import date, timedelta

def build_dim_date(conn, start: str = "2020-01-01", end: str = "2026-12-31") -> None:
    rows = []
    current = datetime.strptime(start, "%Y-%m-%d").date()
    end_date = datetime.strptime(end, "%Y-%m-%d").date()
    date_key = 1

    while current <= end_date:
        rows.append({
            "date_key":   date_key,
            "full_date":  current.strftime("%Y-%m-%d"),
            "year":       current.year,
            "quarter":    (current.month - 1) // 3 + 1,
            "month":      current.month,
            "month_name": current.strftime("%B"),
            "week":       current.isocalendar()[1],
            "day_of_week": current.weekday() + 1,  # 1=Mon, 7=Sun
            "day_name":   current.strftime("%A"),
            "is_weekend": 1 if current.weekday() >= 5 else 0,
        })
        current  += timedelta(days=1)
        date_key += 1

    pd.DataFrame(rows).to_sql("dim_date", conn, if_exists="replace", index=False)
    print(f"dim_date: {len(rows)} rows")

build_dim_date(conn)

In [ ]:
# Load source data
DATA_DIR = Path("cinemastream/data")
users_df  = pd.read_csv(DATA_DIR / "users.csv",        encoding="utf-8", parse_dates=["signup_date"])
movies_df = pd.read_csv(DATA_DIR / "movies.csv",        encoding="utf-8")
events_df = pd.read_csv(DATA_DIR / "watch_events.csv",  encoding="utf-8", parse_dates=["watch_started"])

# Dimension: dim_user (from source users table, enriched)
from datetime import date as date_type

def build_dim_user(users_df: pd.DataFrame, conn) -> None:
    dim = users_df.copy()
    ref_date = datetime(2024, 1, 1).date()
    
    dim["user_key"]   = range(1, len(dim) + 1)  # surrogate key
    dim["tenure_days"] = dim["signup_date"].apply(
        lambda d: (ref_date - d.date()).days if pd.notna(d) else None
    )
    dim["is_churned"]  = dim["churned"].astype(bool)
    dim["plan_monthly_sgd"] = dim["plan"].map({"Free": 0, "Basic": 8.90, "Premium": 12.90})
    
    dim[["user_key", "user_id", "name", "email", "country", "language_pref",
         "plan", "plan_monthly_sgd", "signup_date", "tenure_days", "is_churned"]].to_sql(
        "dim_user", conn, if_exists="replace", index=False
    )
    print(f"dim_user: {len(dim)} rows")

build_dim_user(users_df, conn)

In [ ]:
# Dimension: dim_movie
def build_dim_movie(movies_df: pd.DataFrame, conn) -> None:
    dim = movies_df.copy()
    dim["movie_key"] = range(1, len(dim) + 1)
    dim["is_long_film"] = (dim["runtime_min"] >= 120).astype(int)
    
    dim[["movie_key", "movie_id", "title", "genre", "original_lang",
         "release_year", "runtime_min", "is_long_film"]].to_sql(
        "dim_movie", conn, if_exists="replace", index=False
    )
    print(f"dim_movie: {len(dim)} rows")

build_dim_movie(movies_df, conn)

# Dimension: dim_device (small, static)
def build_dim_device(conn) -> None:
    device_data = [
        {"device_key": 1, "device_type": "Mobile", "device_category": "handheld",  "screen_bucket": "small"},
        {"device_key": 2, "device_type": "TV",     "device_category": "big_screen", "screen_bucket": "large"},
        {"device_key": 3, "device_type": "Tablet",  "device_category": "handheld",  "screen_bucket": "medium"},
        {"device_key": 4, "device_type": "Web",    "device_category": "desktop",   "screen_bucket": "medium"},
    ]
    pd.DataFrame(device_data).to_sql("dim_device", conn, if_exists="replace", index=False)
    print(f"dim_device: 4 rows")

build_dim_device(conn)

In [ ]:
# Fact table: fact_watch_events
def build_fact_watch_events(events_df, users_df, movies_df, conn) -> None:
    fact = events_df.copy()
    
    # Add dimension foreign keys (look up surrogate keys)
    user_key_map  = pd.read_sql("SELECT user_id, user_key FROM dim_user", conn).set_index("user_id")
    movie_key_map = pd.read_sql("SELECT movie_id, movie_key FROM dim_movie", conn).set_index("movie_id")
    date_key_map  = pd.read_sql("SELECT full_date, date_key FROM dim_date", conn).set_index("full_date")
    device_key_map = {"Mobile": 1, "TV": 2, "Tablet": 3, "Web": 4}
    
    fact["user_key"]   = fact["user_id"].map(user_key_map["user_key"])
    fact["movie_key"]  = fact["movie_id"].map(movie_key_map["movie_key"])
    fact["device_key"] = fact["device"].map(device_key_map)
    
    # Date key: extract date from watch_started timestamp
    fact["event_date"] = pd.to_datetime(fact["watch_started"]).dt.strftime("%Y-%m-%d")
    fact["date_key"]   = fact["event_date"].map(date_key_map["date_key"])
    
    # Measures
    fact["completed_flag"] = fact["completed"].astype(int)
    
    # Revenue attribution: only completed sessions get revenue credit
    plan_map = users_df.set_index("user_id")["plan"]
    fact["user_plan"] = fact["user_id"].map(plan_map)
    fact["revenue_attributed_sgd"] = fact.apply(
        lambda row: (8.90 / 30) if row["completed_flag"] and row["user_plan"] == "Basic"
        else (12.90 / 30) if row["completed_flag"] and row["user_plan"] == "Premium"
        else 0.0,
        axis=1
    ).round(4)
    
    # Select fact columns only
    fact_cols = ["event_id", "user_key", "movie_key", "date_key", "device_key",
                 "watch_minutes", "completed_flag", "revenue_attributed_sgd", "country"]
    fact[fact_cols].to_sql("fact_watch_events", conn, if_exists="replace", index=False)
    print(f"fact_watch_events: {len(fact)} rows")

build_fact_watch_events(events_df, users_df, movies_df, conn)

### Querying the Star Schema

In [ ]:
def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

# Analyst query: "What are the top genres by watch time on weekends?"
# Notice: no date arithmetic, no string manipulation — all in the dim tables
genre_weekend = q("""
SELECT
    dm.genre,
    COUNT(*)                            AS sessions,
    SUM(f.watch_minutes)                AS total_minutes,
    ROUND(100.0*SUM(f.completed_flag)/COUNT(*), 1) AS completion_pct
FROM   fact_watch_events f
JOIN   dim_movie dm   ON f.movie_key  = dm.movie_key
JOIN   dim_date  dd   ON f.date_key   = dd.date_key
WHERE  dd.is_weekend = 1
GROUP BY dm.genre
ORDER BY total_minutes DESC
""")
print("Weekend watch stats by genre (star schema query):")
print(genre_weekend)

In [ ]:
# Query with no date arithmetic needed — the dim_date table handles it all
premium_by_quarter = q("""
SELECT
    dd.year,
    dd.quarter,
    du.plan,
    COUNT(*)                            AS sessions,
    ROUND(AVG(f.watch_minutes), 1)      AS avg_min,
    ROUND(SUM(f.revenue_attributed_sgd), 2) AS attributed_rev
FROM   fact_watch_events f
JOIN   dim_user  du ON f.user_key  = du.user_key
JOIN   dim_date  dd ON f.date_key  = dd.date_key
WHERE  du.plan IN ('Premium', 'Basic')
  AND  dd.year = 2023
GROUP BY dd.year, dd.quarter, du.plan
ORDER BY dd.year, dd.quarter, du.plan
""")
print("\nPremium+Basic sessions and attributed revenue by quarter (2023):")
print(premium_by_quarter)

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR = Path("cinemastream/data")
users_df  = pd.read_csv(DATA_DIR / "users.csv",        encoding="utf-8", parse_dates=["signup_date"])
movies_df = pd.read_csv(DATA_DIR / "movies.csv",        encoding="utf-8")
events_df = pd.read_csv(DATA_DIR / "watch_events.csv",  encoding="utf-8", parse_dates=["watch_started"])

conn = sqlite3.connect(":memory:")
build_dim_date(conn)
build_dim_user(users_df, conn)
build_dim_movie(movies_df, conn)
build_dim_device(conn)
build_fact_watch_events(events_df, users_df, movies_df, conn)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
arpu_by_country_quarter = q("""
WITH subscriber_revenue AS (
    SELECT
        du.country,
        dd.year,
        dd.quarter,
        COUNT(DISTINCT f.user_key)              AS active_viewers,
        SUM(f.revenue_attributed_sgd)           AS attributed_rev_sgd
    FROM   fact_watch_events f
    JOIN   dim_user  du ON f.user_key = du.user_key
    JOIN   dim_date  dd ON f.date_key = dd.date_key
    WHERE  du.plan != 'Free'
    GROUP BY du.country, dd.year, dd.quarter
)
SELECT
    country,
    year || '-Q' || quarter AS period,
    active_viewers,
    ROUND(attributed_rev_sgd, 2) AS attributed_rev,
    ROUND(attributed_rev_sgd / NULLIF(active_viewers, 0), 4) AS arpu_sgd
FROM subscriber_revenue
ORDER BY country, period
LIMIT 12
""")
print("=== ARPU by Country and Quarter (Paying Users) ===")
print(arpu_by_country_quarter.to_string(index=False))

In [ ]:
plan_genre_completion = q("""
SELECT
    du.plan,
    dm.genre,
    COUNT(*)                                         AS sessions,
    ROUND(100.0 * SUM(f.completed_flag) / COUNT(*), 1) AS completion_pct
FROM   fact_watch_events f
JOIN   dim_user  du ON f.user_key  = du.user_key
JOIN   dim_movie dm ON f.movie_key = dm.movie_key
GROUP BY du.plan, dm.genre
HAVING COUNT(*) >= 10
ORDER BY completion_pct DESC
LIMIT 8
""")
print("\n=== Plan × Genre Completion Rate (10+ sessions) ===")
print(plan_genre_completion.to_string(index=False))

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
cohort_watch = q("""
SELECT
    CASE
        WHEN du.signup_date LIKE '2022-%' THEN '2022 cohort'
        WHEN du.signup_date LIKE '2023-%' THEN '2023 cohort'
        ELSE 'other'
    END AS cohort,
    COUNT(DISTINCT du.user_key)          AS users_in_cohort,
    COUNT(*)                             AS total_sessions,
    ROUND(AVG(f.watch_minutes), 1)       AS avg_session_min
FROM   fact_watch_events f
JOIN   dim_user du ON f.user_key = du.user_key
GROUP BY cohort
HAVING cohort != 'other'
ORDER BY cohort
""")
print(cohort_watch)

In [ ]:
top3_by_plan = q("""
WITH movie_plan_stats AS (
    SELECT
        du.plan,
        dm.title,
        dm.genre,
        SUM(f.watch_minutes)  AS total_minutes,
        COUNT(*)              AS sessions
    FROM   fact_watch_events f
    JOIN   dim_user  du ON f.user_key  = du.user_key
    JOIN   dim_movie dm ON f.movie_key = dm.movie_key
    GROUP BY du.plan, dm.movie_key, dm.title, dm.genre
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY plan ORDER BY total_minutes DESC) AS plan_rank
    FROM movie_plan_stats
)
SELECT plan, title, genre, total_minutes, sessions, plan_rank
FROM   ranked
WHERE  plan_rank <= 3
ORDER BY plan, plan_rank
""")
print("=== Top 3 Movies per Plan Tier ===")
print(top3_by_plan.to_string(index=False))

---

# Chapter 49: dbt — Transformations as Code

## 0. Where You Are

## 1. The Concept

```
sources (raw warehouse tables)
  → staging models (stg_*: clean, rename, type-cast)
    → intermediate models (int_*: complex joins, deduplication)
      → mart models (dim_*, fct_*: the star schema tables analysts query)
```

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    SRC[sources\nraw warehouse tables] --> STG_U[stg_users.sql]
    SRC --> STG_M[stg_movies.sql]
    SRC --> STG_W[stg_watch_events.sql]
    STG_U --> INT[int_user_sessions.sql]
    STG_W --> INT
    STG_M --> DIM_M[dim_movie.sql]
    STG_U --> DIM_U[dim_user.sql]
    INT --> FCT[fct_watch_events.sql]
    DIM_U --> FCT
    DIM_M --> FCT
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
# dbt is a command-line tool, not a Python library
# Install: pip install dbt-core dbt-duckdb  (or dbt-bigquery, dbt-snowflake)
# These examples show dbt file contents and concepts
# The cinemastream/dbt_cinemastream/ directory is the portfolio project

# Show the dbt project structure
dbt_structure = """
cinemastream/dbt_cinemastream/
├── dbt_project.yml          # project configuration
├── profiles.yml             # connection profiles (not committed to git)
├── models/
│   ├── staging/
│   │   ├── stg_users.sql        # clean + rename raw users
│   │   ├── stg_movies.sql       # clean + rename raw movies
│   │   ├── stg_watch_events.sql # clean + rename raw watch events
│   │   └── schema.yml           # source declarations + column tests
│   ├── intermediate/
│   │   ├── int_user_sessions.sql    # aggregated session stats per user
│   │   └── int_movie_performance.sql # aggregated stats per movie
│   └── marts/
│       ├── dim_user.sql         # final user dimension
│       ├── dim_movie.sql        # final movie dimension
│       ├── fct_watch_events.sql # final fact table
│       └── schema.yml           # mart tests + documentation
├── seeds/
│   └── dim_date.csv             # static date dimension (checked into git)
├── tests/
│   └── assert_no_orphan_events.sql  # custom data test
└── macros/
    └── session_bucket.sql       # reusable SQL macro
"""

print("dbt Project Structure:")
print(dbt_structure)

In [ ]:
# dbt_project.yml — the project configuration
dbt_project_yml = """
name: cinemastream
version: '1.0.0'
config-version: 2

profile: cinemastream    # refers to profiles.yml

model-paths: ["models"]
seed-paths: ["seeds"]
test-paths: ["tests"]
macro-paths: ["macros"]

models:
  cinemastream:
    staging:
      +materialized: view        # staging models are views (fast to refresh)
      +schema: staging
    intermediate:
      +materialized: ephemeral   # intermediate models are CTEs, not stored
    marts:
      +materialized: table       # mart models are tables (fast to query)
      +schema: marts
"""

print("dbt_project.yml content:")
print(dbt_project_yml)

In [ ]:
# Key dbt model files — what you actually write

staging_model = """
-- models/staging/stg_users.sql
-- Staging model: clean and standardise raw users table

WITH source AS (
    SELECT * FROM {{ source('raw', 'users') }}
),

renamed AS (
    SELECT
        user_id,
        TRIM(name)                              AS user_name,
        LOWER(TRIM(email))                      AS email,
        UPPER(TRIM(country))                    AS country,
        LOWER(TRIM(language_pref))              AS language_pref,
        INITCAP(TRIM(plan))                     AS plan,
        CAST(signup_date AS DATE)               AS signup_date,
        CAST(churned AS BOOLEAN)                AS is_churned,
        CASE plan
            WHEN 'Premium' THEN 12.90
            WHEN 'Basic'   THEN  8.90
            ELSE 0.0
        END                                     AS monthly_revenue_sgd
    FROM source
)

SELECT * FROM renamed
"""

mart_dim_user = """
-- models/marts/dim_user.sql
-- User dimension table with surrogate key and tenure

WITH staging AS (
    SELECT * FROM {{ ref('stg_users') }}
),

final AS (
    SELECT
        {{ dbt_utils.generate_surrogate_key(['user_id']) }}  AS user_key,
        user_id,
        user_name,
        email,
        country,
        language_pref,
        plan,
        monthly_revenue_sgd,
        signup_date,
        DATEDIFF('day', signup_date, CURRENT_DATE) AS tenure_days,
        is_churned,
        CURRENT_TIMESTAMP                          AS dbt_updated_at
    FROM staging
)

SELECT * FROM final
"""

fct_watch_events = """
-- models/marts/fct_watch_events.sql
-- Fact table: one row per watch event, with dimension foreign keys

WITH events AS (
    SELECT * FROM {{ ref('stg_watch_events') }}
),

users AS (
    SELECT user_key, user_id, plan FROM {{ ref('dim_user') }}
),

movies AS (
    SELECT movie_key, movie_id FROM {{ ref('dim_movie') }}
),

dates AS (
    SELECT date_key, full_date FROM {{ ref('dim_date') }}
),

final AS (
    SELECT
        e.event_id,
        u.user_key,
        m.movie_key,
        d.date_key,
        CASE e.device
            WHEN 'Mobile' THEN 1
            WHEN 'TV'     THEN 2
            WHEN 'Tablet' THEN 3
            WHEN 'Web'    THEN 4
        END                                      AS device_key,
        e.watch_minutes,
        e.completed                              AS completed_flag,
        e.country                                AS event_country,
        CASE WHEN e.completed AND u.plan = 'Premium' THEN 12.90/30
             WHEN e.completed AND u.plan = 'Basic'   THEN  8.90/30
             ELSE 0.0
        END                                      AS revenue_attributed_sgd
    FROM events e
    LEFT JOIN users  u ON e.user_id  = u.user_id
    LEFT JOIN movies m ON e.movie_id = m.movie_id
    LEFT JOIN dates  d ON DATE(e.watch_started) = d.full_date
)

SELECT * FROM final
"""

print("Key dbt model files shown — see dbt_cinemastream/ for full implementation")
print("\nstg_users.sql (staging):")
print(staging_model[:400], "...")

In [ ]:
# dbt test configuration — schema.yml
schema_yml = """
version: 2

models:
  - name: stg_users
    description: "Cleaned and renamed CinemaStream users from raw source"
    columns:
      - name: user_id
        description: "Natural key from source system"
        tests:
          - not_null
          - unique
      - name: plan
        description: "Subscription plan: Free, Basic, or Premium"
        tests:
          - not_null
          - accepted_values:
              values: ['Free', 'Basic', 'Premium']
      - name: country
        description: "2-char ISO country code"
        tests:
          - not_null
          - accepted_values:
              values: ['SG', 'MY', 'ID', 'PH', 'TH', 'VN', 'IN']
      - name: email
        tests:
          - not_null
          - unique

  - name: fct_watch_events
    description: "Fact table: one row per watch event"
    columns:
      - name: event_id
        tests:
          - not_null
          - unique
      - name: user_key
        tests:
          - not_null
          - relationships:
              to: ref('dim_user')
              field: user_key
      - name: watch_minutes
        tests:
          - not_null
"""

print("schema.yml (tests and documentation):")
print(schema_yml[:500], "...")

In [ ]:
# Simulating dbt model execution without dbt installed
# Demonstrates the same transformation logic

import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR = Path("cinemastream/data")
users_df  = pd.read_csv(DATA_DIR / "users.csv",        encoding="utf-8")
movies_df = pd.read_csv(DATA_DIR / "movies.csv",        encoding="utf-8")
events_df = pd.read_csv(DATA_DIR / "watch_events.csv",  encoding="utf-8")

conn = sqlite3.connect(":memory:")
users_df.to_sql("raw_users",        conn, index=False, if_exists="replace")
movies_df.to_sql("raw_movies",       conn, index=False, if_exists="replace")
events_df.to_sql("raw_watch_events", conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)
def execute(sql: str) -> None:
    conn.execute(sql); conn.commit()

# "dbt run" equivalent: create each model in dependency order
print("[dbt run] Creating stg_users...")
execute("""
CREATE TABLE stg_users AS
SELECT
    user_id,
    TRIM(name)      AS user_name,
    LOWER(email)    AS email,
    UPPER(country)  AS country,
    language_pref,
    plan,
    signup_date,
    CAST(churned AS INTEGER)  AS is_churned,
    CASE plan
        WHEN 'Premium' THEN 12.90
        WHEN 'Basic'   THEN  8.90
        ELSE 0.0
    END AS monthly_revenue_sgd
FROM raw_users
""")

print("[dbt run] Creating stg_movies...")
execute("""
CREATE TABLE stg_movies AS
SELECT
    movie_id,
    TRIM(title)    AS title,
    genre,
    original_lang,
    release_year,
    runtime_min,
    CAST(runtime_min >= 120 AS INTEGER) AS is_long_film
FROM raw_movies
""")

print("[dbt run] Creating stg_watch_events...")
execute("""
CREATE TABLE stg_watch_events AS
SELECT
    event_id,
    user_id,
    movie_id,
    watch_started,
    CASE
        WHEN watch_minutes IS NULL OR watch_minutes < 0 THEN NULL
        WHEN watch_minutes > 300 THEN 300
        ELSE watch_minutes
    END AS watch_minutes,
    CAST(completed AS INTEGER) AS completed,
    TRIM(device)  AS device,
    UPPER(country) AS country
FROM raw_watch_events
""")

print("[dbt run] All staging models created.")
print(f"\nStaging row counts:")
for t in ["stg_users", "stg_movies", "stg_watch_events"]:
    n = q(f"SELECT COUNT(*) AS n FROM {t}")['n'][0]
    print(f"  {t}: {n} rows")

In [ ]:
# "dbt test" equivalent: run data quality assertions
print("\n[dbt test] Running generic tests...")

test_results = []

# not_null test on user_id
null_user_ids = q("SELECT COUNT(*) AS n FROM stg_users WHERE user_id IS NULL")['n'][0]
test_results.append(("stg_users.user_id.not_null", null_user_ids == 0, null_user_ids))

# unique test on user_id
dup_user_ids = q("SELECT COUNT(*) AS n FROM (SELECT user_id FROM stg_users GROUP BY user_id HAVING COUNT(*) > 1)")['n'][0]
test_results.append(("stg_users.user_id.unique", dup_user_ids == 0, dup_user_ids))

# accepted_values test on plan
invalid_plans = q("SELECT COUNT(*) AS n FROM stg_users WHERE plan NOT IN ('Free','Basic','Premium')")['n'][0]
test_results.append(("stg_users.plan.accepted_values", invalid_plans == 0, invalid_plans))

# not_null on event_id
null_events = q("SELECT COUNT(*) AS n FROM stg_watch_events WHERE event_id IS NULL")['n'][0]
test_results.append(("stg_watch_events.event_id.not_null", null_events == 0, null_events))

# range test (custom): watch_minutes
bad_mins = q("SELECT COUNT(*) AS n FROM stg_watch_events WHERE watch_minutes NOT BETWEEN 1 AND 300")['n'][0]
test_results.append(("stg_watch_events.watch_minutes.range", bad_mins == 0, bad_mins))

all_pass = all(passed for _, passed, _ in test_results)
for test_name, passed, count in test_results:
    status = "pass" if passed else "FAIL"
    print(f"  [{status}] {test_name} (failures: {count})")

print(f"\n[dbt test] {'All tests passed.' if all_pass else 'FAILURES detected — check above'}")

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

# Setup (same as above)
DATA_DIR = Path("cinemastream/data")
users_df  = pd.read_csv(DATA_DIR / "users.csv",        encoding="utf-8")
movies_df = pd.read_csv(DATA_DIR / "movies.csv",        encoding="utf-8")
events_df = pd.read_csv(DATA_DIR / "watch_events.csv",  encoding="utf-8")

conn = sqlite3.connect(":memory:")

# Apply the same staging transform as stg_users.sql so the mart SQL below
# sees the renamed/derived columns (user_name, is_churned, monthly_revenue_sgd).
stg_users_df = users_df.assign(
    user_name=users_df["name"].str.strip(),
    is_churned=users_df["churned"].astype(int),
    monthly_revenue_sgd=users_df["plan"].map(
        {"Premium": 12.90, "Basic": 8.90}).fillna(0.0),
)
stg_users_df.to_sql("stg_users",     conn, index=False, if_exists="replace")
movies_df.to_sql("stg_movies",       conn, index=False, if_exists="replace")
events_df.to_sql("stg_watch_events", conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)
def execute(sql: str) -> None:
    conn.execute(sql); conn.commit()

In [ ]:
# This is what fct_user_weekly_summary.sql would contain
execute("""
CREATE TABLE mart_user_weekly_summary AS
WITH user_stats AS (
    SELECT
        we.user_id,
        COUNT(*)                               AS sessions_this_week,
        SUM(we.watch_minutes)                  AS total_watch_minutes,
        ROUND(100.0*SUM(we.completed)/COUNT(*), 1) AS completion_rate,
        COUNT(DISTINCT we.movie_id)            AS distinct_movies
    FROM stg_watch_events we
    GROUP BY we.user_id
),
users AS (
    SELECT user_id, user_name, plan, country, is_churned, monthly_revenue_sgd
    FROM stg_users
    WHERE is_churned = 0
)
SELECT
    u.user_id,
    u.user_name,
    u.plan,
    u.country,
    u.monthly_revenue_sgd,
    COALESCE(us.sessions_this_week, 0)   AS sessions,
    COALESCE(us.total_watch_minutes, 0)  AS total_minutes,
    COALESCE(us.completion_rate, 0)      AS completion_rate,
    COALESCE(us.distinct_movies, 0)      AS distinct_movies,
    CASE WHEN us.user_id IS NULL THEN 1 ELSE 0 END AS is_inactive
FROM      users u
LEFT JOIN user_stats us ON u.user_id = us.user_id
""")

summary = q("""
SELECT 
    is_inactive,
    plan,
    COUNT(*) AS users,
    ROUND(AVG(total_minutes), 1) AS avg_minutes,
    ROUND(AVG(completion_rate), 1) AS avg_completion
FROM mart_user_weekly_summary
GROUP BY is_inactive, plan
ORDER BY is_inactive, plan
""")
print("mart_user_weekly_summary — active vs inactive by plan:")
print(summary.to_string(index=False))

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
staging_movies_sql = """
-- models/staging/stg_movies.sql
WITH source AS (
    SELECT * FROM {{ source('raw', 'movies') }}
),

cleaned AS (
    SELECT
        movie_id,
        TRIM(title)                 AS title,
        genre,
        original_lang,
        CAST(release_year AS INT)   AS release_year,
        runtime_min,
        CASE
            WHEN release_year < 2000  THEN 'classic'
            WHEN release_year <= 2015 THEN 'modern'
            ELSE 'contemporary'
        END                         AS era
    FROM source
)

SELECT * FROM cleaned
"""

# Simulate execution
execute("""
CREATE TABLE stg_movies_v2 AS
SELECT
    movie_id,
    TRIM(title)     AS title,
    genre,
    original_lang,
    CAST(release_year AS INTEGER) AS release_year,
    runtime_min,
    CASE
        WHEN release_year < 2000  THEN 'classic'
        WHEN release_year <= 2015 THEN 'modern'
        ELSE 'contemporary'
    END AS era
FROM stg_movies
""")
result = q("SELECT movie_id, title, release_year, era FROM stg_movies_v2 LIMIT 5")
print("stg_movies with era column:")
print(result)

In [ ]:
schema_yml_answer = """
version: 2

models:
  - name: stg_watch_events
    description: "Cleaned CinemaStream watch events — one row per viewing session"
    columns:
      - name: event_id
        description: "Unique identifier for each watch event"
        tests:
          - not_null
          - unique

      - name: watch_minutes
        description: "Duration of the viewing session in minutes (1–300)"
        tests:
          - not_null

      - name: device
        description: "Device type used for viewing"
        tests:
          - accepted_values:
              values: ['Mobile', 'TV', 'Tablet', 'Web']

      - name: country
        description: "ISO 2-char country code of the watch event location"
        tests:
          - accepted_values:
              values: ['SG', 'MY', 'ID', 'PH', 'TH', 'VN', 'IN']
"""

print("schema.yml for stg_watch_events:")
print(schema_yml_answer)

In [ ]:
execute("""
CREATE TABLE mart_monthly_revenue AS
WITH user_months AS (
    -- cross join users with a list of months they were active
    -- simplified: use signup_date to approximate
    SELECT
        u.user_id,
        u.plan,
        u.monthly_revenue_sgd,
        SUBSTR(u.signup_date, 1, 7) AS active_since_month
    FROM stg_users u
    WHERE u.is_churned = 0
)
SELECT
    SUBSTR(active_since_month, 1, 4) AS year,
    SUBSTR(active_since_month, 6, 2) AS month,
    plan,
    COUNT(*) AS active_users,
    ROUND(SUM(monthly_revenue_sgd), 2) AS monthly_revenue_sgd
FROM user_months
GROUP BY year, month, plan
ORDER BY year, month, plan
""")

result = q("SELECT * FROM mart_monthly_revenue LIMIT 9")
print("mart_monthly_revenue (first 9 rows):")
print(result.to_string(index=False))

---

# Chapter 50: Cloud Data Warehouses — BigQuery, Snowflake, Redshift

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart LR
    subgraph OLTP["Operational DB (OLTP)\nPostgreSQL / MySQL"]
        R["Row-oriented\noptimised for writes\nand point lookups"]
    end
    subgraph ETL["ETL / ELT"]
        E["Extract"] --> T["Transform"] --> L["Load"]
    end
    subgraph DW["Cloud Data Warehouse (OLAP)"]
        COL["Columnar storage\nMassively parallel\nSeparate storage + compute"]
        BQ["BigQuery\n(GCP · per-TB)"]
        SF["Snowflake\n(multi-cloud · per-credit)"]
        RS["Redshift\n(AWS · per-node)"]
        COL --> BQ & SF & RS
    end
    subgraph BI["Consumers"]
        SQL["Analyst SQL"]
        DBT["dbt models"]
        DASH["Dashboard\n(Streamlit / Looker)"]
    end

    OLTP -->|"CDC / batch export"| ETL --> DW --> BI
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install duckdb pandas

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR = Path("cinemastream/data")

### BigQuery — The Key Concepts

In [ ]:
# BigQuery: serverless, pay-per-query, no cluster to manage

bigquery_concepts = {
    "billing_model": "You pay per TB of data scanned, not per query count or time. "
                     "SELECT COUNT(*) costs almost nothing. SELECT * on a 10TB table is expensive.",
    
    "partitioning": "Partition tables by date column (DATE(watch_started)). "
                    "A query with WHERE DATE(watch_started) BETWEEN '2023-01-01' AND '2023-01-31' "
                    "scans only January's partition — 1/12 of the data.",
    
    "clustering": "Within partitions, cluster by frequently-filtered columns (user_id, country). "
                  "Queries like WHERE user_id = 42 only read the relevant data blocks.",
    
    "slot_reservations": "By default, you share compute with other BigQuery users (on-demand). "
                         "For predictable latency, purchase dedicated slots (commitments).",
    
    "streaming_insert": "Real-time data can be streamed directly to BigQuery via the "
                        "Storage Write API — no intermediate files needed.",
    
    "bq_storage_api": "Read BigQuery tables directly into Python pandas/Polars with "
                      "google-cloud-bigquery-storage — much faster than downloading CSV.",
}

print("BigQuery Key Concepts:")
for k, v in bigquery_concepts.items():
    print(f"\n  {k}:")
    print(f"    {v}")

In [ ]:
# BigQuery API pattern (requires: pip install google-cloud-bigquery pandas-gbq)
bigquery_python = '''
from google.cloud import bigquery
import pandas as pd

# Authenticate: uses Application Default Credentials (run: gcloud auth application-default login)
client = bigquery.Client(project="cinemastream-data")

# Run a query → DataFrame
query = """
    SELECT 
        DATE_TRUNC(watch_started, MONTH)  AS month,
        plan,
        COUNT(*)                          AS sessions,
        ROUND(AVG(watch_minutes), 1)      AS avg_min
    FROM `cinemastream-data.analytics.watch_events`
    WHERE watch_started >= TIMESTAMP('2023-01-01')
    GROUP BY month, plan
    ORDER BY month, plan
"""

df = client.query(query).to_dataframe()
print(df)

# Table metadata
table = client.get_table("cinemastream-data.analytics.watch_events")
print(f"Table: {table.full_table_id}")
print(f"Rows: {table.num_rows:,}")
print(f"Storage: {table.num_bytes / 1e9:.1f} GB")
'''

print("BigQuery Python API (requires google-cloud-bigquery):")
print(bigquery_python)

### Snowflake — The Key Concepts

In [ ]:
snowflake_concepts = {
    "virtual_warehouses": "Compute is abstracted as 'virtual warehouses' (XS, S, M, L, XL). "
                          "You can scale up for heavy queries and pause when idle. "
                          "Multiple warehouses can share the same storage simultaneously.",
    
    "time_travel":        "Snowflake stores previous versions of every table for up to 90 days. "
                          "SELECT * FROM table AT(TIMESTAMP => '2024-01-01') lets you "
                          "query historical snapshots — critical for debugging bad loads.",
    
    "data_sharing":       "Share tables with other Snowflake accounts without copying data. "
                          "External companies can query your tables live — the 'Data Exchange' feature.",
    
    "snowpipe":           "Snowflake's micro-batch streaming ingestion — triggers loads "
                          "automatically when new files arrive in S3/GCS/Azure Blob.",
    
    "zero_copy_clone":    "Clone a production table instantly (no data copy). "
                          "Perfect for staging environments and developer sandboxes.",
}

print("Snowflake Key Concepts:")
for k, v in snowflake_concepts.items():
    print(f"\n  {k}:")
    print(f"    {v}")

### Simulating Warehouse Patterns Locally with DuckDB

In [ ]:
# DuckDB is the local development warehouse — same SQL dialect as BigQuery/Snowflake
# Perfect for testing dbt models and warehouse patterns locally
# pip install duckdb

try:
    import duckdb
    DUCKDB_AVAILABLE = True
except ImportError:
    DUCKDB_AVAILABLE = False

if DUCKDB_AVAILABLE:
    conn_duck = duckdb.connect(":memory:")
    
    # DuckDB can query Parquet/CSV files directly — no LOAD step
    conn_duck.execute("""
    CREATE TABLE watch_events AS 
    SELECT * FROM read_csv_auto('cinemastream/data/watch_events.csv')
    """)
    
    result = conn_duck.execute("""
    SELECT 
        country,
        COUNT(*)                 AS sessions,
        ROUND(AVG(watch_minutes), 1) AS avg_min
    FROM watch_events
    GROUP BY country
    ORDER BY sessions DESC, country
    """).df()
    
    print("DuckDB query result:")
    print(result)
else:
    print("DuckDB not installed — showing expected output")
    expected = pd.DataFrame({
        "country": ["ID", "MY", "VN", "TH", "SG", "IN", "PH"],
        "sessions": [102, 80, 64, 35, 33, 31, 25],
        "avg_min": [71.8, 76.4, 72.6, 77.1, 78.9, 72.4, 71.8],
    })
    print(expected)

### Cost Optimization Patterns

In [ ]:
cost_patterns = [
    {
        "pattern": "Partition pruning",
        "bad":     "SELECT * FROM events WHERE DATE(watch_started) = '2023-01-15'",
        "good":    "SELECT * FROM events WHERE watch_started BETWEEN TIMESTAMP('2023-01-15') AND TIMESTAMP('2023-01-16')",
        "reason":  "DATE() function prevents partition pruning in BigQuery. Date range condition uses partitions.",
    },
    {
        "pattern": "Column selection",
        "bad":     "SELECT * FROM events",
        "good":    "SELECT event_id, user_id, watch_minutes FROM events",
        "reason":  "Columnar storage: SELECT * scans all columns. Select only what you need.",
    },
    {
        "pattern": "Avoid LIMIT for cost",
        "bad":     "SELECT * FROM events LIMIT 100",
        "good":    "SELECT * FROM events LIMIT 100  -- (note: BigQuery still scans all data before limiting)",
        "reason":  "In BigQuery, LIMIT reduces output but NOT scan cost. Add WHERE clause to reduce scan.",
    },
    {
        "pattern": "Cache queries",
        "bad":     "Running the same query 100× a day",
        "good":    "Save results to a table; query the table instead",
        "reason":  "BigQuery caches results for 24h for identical queries (same SQL + same tables). Explicit result tables are faster and cheaper for dashboards.",
    },
]

print("Warehouse Cost Optimisation Patterns:")
for i, p in enumerate(cost_patterns, 1):
    print(f"\n{i}. {p['pattern']}")
    print(f"   Bad:  {p['bad']}")
    print(f"   Good: {p['good']}")
    print(f"   Why:  {p['reason']}")

## 3. CinemaStream in Practice

In [ ]:
cinemastream_architecture = {
    "warehouse": "BigQuery",
    "datasets": {
        "raw": {
            "description": "Raw ingested tables — append-only, immutable",
            "tables":      ["raw.watch_events", "raw.users", "raw.movies", "raw.subscriptions"],
            "access":      "Data engineers only (ingestion pipelines)",
            "retention":   "90 days (cost control)",
        },
        "staging": {
            "description": "dbt staging models — views over raw tables",
            "tables":      ["staging.stg_watch_events", "staging.stg_users", "staging.stg_movies"],
            "access":      "Data engineers, dbt runs",
            "retention":   "Views — no storage cost",
        },
        "marts": {
            "description": "Final star schema — stable, tested, documented",
            "tables":      ["marts.fct_watch_events", "marts.dim_user", "marts.dim_movie",
                           "marts.dim_date", "marts.mart_weekly_metrics"],
            "access":      "All analysts, Looker/Metabase dashboards, ML team",
            "retention":   "Indefinite",
        },
    },
    "ingestion_tool": "Fivetran (users, movies) + custom Airflow DAG (watch_events)",
    "transformation_tool": "dbt Cloud (runs daily at 02:30 UTC)",
    "BI_tool":            "Looker (queries marts layer only)",
}

print("=== CinemaStream BigQuery Architecture ===")
for dataset_name, details in cinemastream_architecture["datasets"].items():
    print(f"\n  Dataset: {dataset_name}")
    print(f"    {details['description']}")
    print(f"    Access: {details['access']}")
    print(f"    Tables: {', '.join(details['tables'][:3])}")

In [ ]:
# Cost estimation for CinemaStream's warehouse usage
import pandas as pd

# BigQuery pricing: $5 per TB scanned (on-demand, Sep 2024)
BQ_PRICE_PER_TB = 5.0

scenarios = [
    {
        "query":          "Daily metrics dashboard (full day scan)",
        "table_size_gb":  0.1,   # 381 events = tiny; production would be 100GB/day
        "queries_per_day": 20,
        "columns_selected": 5,
        "total_columns":   8,
    },
    {
        "query":          "Weekly board metrics query",
        "table_size_gb":  0.7,
        "queries_per_day": 1,
        "columns_selected": 6,
        "total_columns":   8,
    },
    {
        "query":          "ML feature extraction (full history scan)",
        "table_size_gb":  10.0,
        "queries_per_day": 0.14,  # once per week
        "columns_selected": 8,
        "total_columns":   8,
    },
]

print("BigQuery Cost Estimates (monthly):\n")
total_monthly_cost = 0
for s in scenarios:
    # Columnar storage: only columns_selected/total_columns of data scanned
    effective_gb = s["table_size_gb"] * (s["columns_selected"] / s["total_columns"])
    monthly_tb   = effective_gb / 1024 * s["queries_per_day"] * 30
    monthly_cost = monthly_tb * BQ_PRICE_PER_TB
    total_monthly_cost += monthly_cost
    print(f"  {s['query']}")
    print(f"    Scanned: {effective_gb:.2f}GB per query × {s['queries_per_day']*30:.0f} runs/month")
    print(f"    Cost: ${monthly_cost:.2f}/month")

print(f"\nTotal estimated monthly BigQuery cost: ${total_monthly_cost:.2f}")
print("(At 5M users / 100M events/month, scale these estimates × 260,000)")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
# Without partition filter
full_table_tb    = 5.0
price_per_tb     = 5.0
columns_selected = 2
total_columns    = 10

# Columnar: 2/10 columns scanned
scanned_tb = full_table_tb * (columns_selected / total_columns)
cost       = scanned_tb * price_per_tb
print(f"No partition filter: scans {scanned_tb:.2f}TB → costs ${cost:.2f}")

# With partition filter (1 day of 365)
scanned_with_partition = scanned_tb / 365
cost_with_partition    = scanned_with_partition * price_per_tb
print(f"With date partition (1 day): scans {scanned_with_partition:.4f}TB → costs ${cost_with_partition:.4f}")

# Better query
better_query = """
SELECT user_id, watch_minutes
FROM   watch_events
WHERE  DATE(watch_started) = '2023-01-15'    -- partition column: prunes 364/365 of data
  AND  country = 'SG'               -- further filter after partition pruning
"""
print(f"\nBetter query:\n{better_query}")

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

# Raw table: has data quality issues
raw_data = pd.DataFrame({
    "user_id": [1, 1, 2, 3],          # user 1 appears twice (duplicate)
    "plan":    ["premium", "Premium", "BASIC", "Free"],  # mixed casing
    "country": ["sg", "SG", "MY", "IN"],               # mixed casing
})
raw_data.to_sql("raw_users", conn, index=False)

# Marts table: cleaned by dbt
marts_data = pd.DataFrame({
    "user_id": [1, 2, 3],             # deduplicated
    "plan":    ["Premium", "Basic", "Free"],           # standardised
    "country": ["SG", "MY", "IN"],                    # standardised
})
marts_data.to_sql("marts_dim_user", conn, index=False)

# Wrong result from raw: overcounts user 1, wrong plan case
raw_result = pd.read_sql("""
SELECT plan, COUNT(*) AS users
FROM raw_users
GROUP BY plan
""", conn)

# Correct result from marts
marts_result = pd.read_sql("""
SELECT plan, COUNT(*) AS users
FROM marts_dim_user
GROUP BY plan
""", conn)

print("Query from RAW table (wrong — 'Premium' vs 'premium' = 2 rows each):")
print(raw_result)
print("\nQuery from MARTS table (correct):")
print(marts_result)

---

# Chapter 51: Data Governance, Lineage, and the Analytics Engineer Role

## 0. Where You Are

## 1. The Concept

### Data Governance

### Data Lineage

### The Analytics Engineer Role

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart LR
    RAW[raw.watch_events\nPostgres source] --> STG[stg_watch_events\ndbt staging]
    RAW2[raw.users] --> STG2[stg_users\ndbt staging]
    STG --> FCT[fct_watch_events\ndbt mart]
    STG2 --> FCT
    FCT --> RPT[mart_monthly_revenue\ndbt mart]
    RPT --> DASH[Dashboard\nanalyst query]
    DASH -. impact analysis .-> RAW
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### Building a Lineage Graph

In [ ]:
from collections import defaultdict, deque

# A simplified lineage graph: node -> list of downstream nodes it feeds
lineage_graph = {
    "raw.orders":          ["stg_orders"],
    "raw.customers":       ["stg_customers"],
    "raw.products":        ["stg_products"],
    "stg_orders":          ["fct_orders", "int_order_items"],
    "stg_customers":       ["dim_customer"],
    "stg_products":        ["dim_product", "int_order_items"],
    "int_order_items":     ["fct_orders"],
    "fct_orders":          ["mart_revenue_by_month", "mart_customer_ltv"],
    "dim_customer":        ["mart_customer_ltv"],
    "dim_product":         ["mart_revenue_by_product"],
    "mart_revenue_by_month": [],
    "mart_customer_ltv":     [],
    "mart_revenue_by_product": [],
}

def downstream_of(graph, node):
    """BFS to find every node downstream of `node` (impact analysis)."""
    visited = set()
    queue = deque([node])
    while queue:
        current = queue.popleft()
        for neighbor in graph.get(current, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return visited

# Impact analysis: what breaks if raw.orders changes?
impacted = downstream_of(lineage_graph, "raw.orders")
print("If raw.orders schema changes, these models are impacted:")
for node in sorted(impacted):
    print(f"  - {node}")

### Upstream Lineage (Where Did This Come From?)

In [ ]:
def upstream_of(graph, target):
    """Find every node that (transitively) feeds into `target`."""
    reverse_graph = defaultdict(list)
    for node, downstream_list in graph.items():
        for downstream in downstream_list:
            reverse_graph[downstream].append(node)
    
    visited = set()
    queue = deque([target])
    while queue:
        current = queue.popleft()
        for upstream in reverse_graph.get(current, []):
            if upstream not in visited:
                visited.add(upstream)
                queue.append(upstream)
    return visited

# Debugging: mart_customer_ltv looks wrong. What feeds it?
sources = upstream_of(lineage_graph, "mart_customer_ltv")
print("mart_customer_ltv is built from:")
for node in sorted(sources):
    print(f"  - {node}")

### Data Classification and Access Tiers

In [ ]:
# A simple data classification scheme
data_classification = {
    "public": {
        "description": "No restrictions. Safe for any employee or external partner.",
        "examples":    ["movie_id, title, genre, release_year (movie catalog)"],
        "access":      "Everyone",
    },
    "internal": {
        "description": "Internal business data. Not PII, but not for public release.",
        "examples":    ["watch_events (aggregated), country-level subscriber counts"],
        "access":      "All employees",
    },
    "confidential": {
        "description": "Sensitive business data — financial, strategic.",
        "examples":    ["MRR, churn rate, individual revenue figures"],
        "access":      "Finance, leadership, relevant analysts",
    },
    "restricted_pii": {
        "description": "Personally identifiable information. Legal/compliance requirements apply.",
        "examples":    ["users.email, users.full_name, users.payment_method"],
        "access":      "Named individuals only, with audit logging. Masked in non-prod.",
    },
}

print("CinemaStream Data Classification Tiers:\n")
for tier, details in data_classification.items():
    print(f"  {tier.upper()}")
    print(f"    {details['description']}")
    print(f"    Examples: {details['examples'][0]}")
    print(f"    Access: {details['access']}\n")

### PII Masking for Non-Production Environments

In [ ]:
import hashlib
import pandas as pd

def mask_pii(df: pd.DataFrame, pii_columns: list[str]) -> pd.DataFrame:
    """
    Mask PII columns for use in dev/staging environments.
    Email -> consistent hash (same input = same output, preserves join keys)
    Names -> generic placeholder with row index
    """
    masked = df.copy()
    for col in pii_columns:
        if col not in masked.columns:
            continue
        if "email" in col.lower():
            masked[col] = masked[col].apply(
                lambda x: hashlib.sha256(str(x).encode()).hexdigest()[:10] + "@masked.test"
            )
        elif "name" in col.lower():
            masked[col] = [f"User_{i:04d}" for i in range(len(masked))]
        else:
            masked[col] = "***MASKED***"
    return masked

# Demo
sample_users = pd.DataFrame({
    "user_id":   [1, 2, 3],
    "full_name": ["Ravi Kumar", "Siti Rahman", "Nguyen Van Minh"],
    "email":     ["ravi@example.com", "siti@example.com", "minh@example.com"],
    "country":   ["IN", "MY", "VN"],
    "plan":      ["Premium", "Basic", "Free"],
})

masked_users = mask_pii(sample_users, pii_columns=["full_name", "email"])
print("Original (production):")
print(sample_users)
print("\nMasked (for dev/staging):")
print(masked_users)

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR = Path("cinemastream/data")
conn = sqlite3.connect(":memory:")

users = pd.read_csv(DATA_DIR / "users.csv")
# The raw export calls the column `churned`; the governed schema standardises it
# to `is_churned` (0/1). Rename on load so the MRR queries below match the schema.
users = users.rename(columns={"churned": "is_churned"})
users["is_churned"] = users["is_churned"].astype(int)
users.to_sql("users", conn, index=False, if_exists="replace")

# CinemaStream's lineage for the MRR metric
mrr_lineage = {
    "raw.subscriptions":   ["stg_subscriptions"],
    "stg_subscriptions":   ["fct_subscription_events", "dim_user"],
    "fct_subscription_events": ["mart_monthly_revenue"],
    "dim_user":            ["mart_monthly_revenue"],
    "mart_monthly_revenue": ["dashboard_finance_mrr", "dashboard_data_team_mrr"],
    "dashboard_finance_mrr":    [],
    "dashboard_data_team_mrr":  [],
}

# Both dashboards trace back to mart_monthly_revenue — so the divergence
# must be introduced AFTER that point, in each dashboard's own query

def upstream_of(graph, target):
    from collections import defaultdict, deque
    reverse_graph = defaultdict(list)
    for node, downs in graph.items():
        for d in downs:
            reverse_graph[d].append(node)
    visited, queue = set(), deque([target])
    while queue:
        cur = queue.popleft()
        for up in reverse_graph.get(cur, []):
            if up not in visited:
                visited.add(up)
                queue.append(up)
    return visited

print("Lineage of dashboard_finance_mrr:")
print(sorted(upstream_of(mrr_lineage, "dashboard_finance_mrr")))
print("\nLineage of dashboard_data_team_mrr:")
print(sorted(upstream_of(mrr_lineage, "dashboard_data_team_mrr")))
print("\n--> Both dashboards share the SAME upstream lineage through mart_monthly_revenue.")
print("--> The discrepancy must be in each dashboard's own query logic, not the data model.")

In [ ]:
# The data team's MRR query (correct — Ch 038's verified definition)
data_team_mrr = pd.read_sql("""
SELECT
    ROUND(SUM(
        CASE
            WHEN plan = 'Premium' AND is_churned = 0 THEN 12.90
            WHEN plan = 'Basic'   AND is_churned = 0 THEN 8.90
            ELSE 0
        END
    ), 2) AS mrr
FROM users
""", conn)

# Finance's spreadsheet formula (WRONG — counts ALL Basic/Premium users,
# including churned ones, because the spreadsheet was built before is_churned existed)
finance_mrr_wrong = pd.read_sql("""
SELECT
    ROUND(SUM(
        CASE
            WHEN plan = 'Premium' THEN 12.90
            WHEN plan = 'Basic'   THEN 8.90
            ELSE 0
        END
    ), 2) AS mrr
FROM users
""", conn)

print("Data team MRR (excludes churned):", data_team_mrr["mrr"].iloc[0])
print("Finance MRR (includes churned — BUG):", finance_mrr_wrong["mrr"].iloc[0])
print(f"\nDifference: {finance_mrr_wrong['mrr'].iloc[0] - data_team_mrr['mrr'].iloc[0]:.2f}")
print("\nRoot cause: Finance's spreadsheet was built before `is_churned` existed in the schema.")
print("It was never updated. Lineage shows both pull from the same mart — the bug is")
print("downstream of the model, in a manually-maintained spreadsheet formula.")

In [ ]:
# CinemaStream's data ownership matrix (governance document)
ownership_matrix = pd.DataFrame([
    {"table":         "raw.users",                  "owner": "Carlos (Backend)",   "classification": "restricted_pii", "downstream_consumers": "stg_users only"},
    {"table":         "raw.watch_events",           "owner": "Priya (Data Eng)",   "classification": "internal",       "downstream_consumers": "stg_watch_events"},
    {"table":         "marts.dim_user",             "owner": "Priya (Analytics Eng)", "classification": "confidential", "downstream_consumers": "All marts, ML features"},
    {"table":         "marts.mart_monthly_revenue", "owner": "Priya (Analytics Eng)", "classification": "confidential", "downstream_consumers": "Looker (Finance, Leadership)"},
    {"table":         "marts.fct_watch_events",     "owner": "Priya (Analytics Eng)", "classification": "internal",       "downstream_consumers": "Looker (Product), ML features"},
])
print(ownership_matrix.to_string(index=False))

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
graph = {
    "raw.products":  ["stg_products"],
    "stg_products":  ["dim_product", "int_order_items"],
    "int_order_items": ["fct_orders"],
    "fct_orders":    ["mart_revenue_by_month"],
    "dim_product":   ["mart_revenue_by_product"],
    "mart_revenue_by_month":   [],
    "mart_revenue_by_product": [],
}

In [ ]:
from collections import deque

def downstream_of(graph, node):
    visited, queue = set(), deque([node])
    while queue:
        cur = queue.popleft()
        for nxt in graph.get(cur, []):
            if nxt not in visited:
                visited.add(nxt)
                queue.append(nxt)
    return visited

graph = {
    "raw.products":  ["stg_products"],
    "stg_products":  ["dim_product", "int_order_items"],
    "int_order_items": ["fct_orders"],
    "fct_orders":    ["mart_revenue_by_month"],
    "dim_product":   ["mart_revenue_by_product"],
    "mart_revenue_by_month":   [],
    "mart_revenue_by_product": [],
}

impacted = downstream_of(graph, "stg_products")
print("Impacted by stg_products breaking:")
for node in sorted(impacted):
    print(f"  - {node}")

marts_impacted = [n for n in impacted if n.startswith("mart_")]
print(f"\nMarts impacted: {marts_impacted}")
print("BOTH marts break — stg_products feeds dim_product directly AND")
print("int_order_items -> fct_orders -> mart_revenue_by_month.")

In [ ]:
classifications = {
    "movies.title":
        ("public", "Movie titles are shown to all users on the platform — no restriction needed."),
    "users.payment_method":
        ("restricted_pii", "Payment details are sensitive financial PII — strictly access-controlled and audited."),
    "mart_monthly_revenue.total_revenue":
        ("confidential", "Company financial performance — restricted to finance/leadership/relevant analysts."),
    "fct_watch_events.watch_minutes (aggregated, no user_id)":
        ("internal", "Useful for product analytics, not PII once aggregated, but not for public release (competitive info)."),
}

for item, (tier, reason) in classifications.items():
    print(f"{item}\n  -> {tier.upper()}: {reason}\n")

In [ ]:
policy = """
GOVERNANCE POLICY: marts.dim_user
-----------------------------------
Owner: Priya (Analytics Engineering) — first point of contact for schema
       questions, quality issues, or access requests.

Classification: CONFIDENTIAL. This table contains plan tier, country,
       tenure, and engagement metrics per user (no email/name — those
       remain in raw.users, restricted_pii). Confidential because plan
       and revenue data are commercially sensitive.

Default access: Data team, Product team (Rohan + team), ML team (Mei),
       and Finance (Dharani's team) via Looker. Access is via the marts
       dataset only — direct BigQuery access requires a documented
       business justification.

Access requests: Anyone outside the default list (e.g., a new hire,
       an external contractor, a partner company) must submit a written
       request to Priya stating purpose and duration. Time-limited
       access is granted via IAM with an expiry date. All access is
       logged. Requests for raw.users (PII) require additional sign-off
       from Carlos (data owner) and are logged separately with longer
       retention for audit purposes.
"""
print(policy)

---

# Chapter 51a: Dataset Cards, Model Cards & AI Documentation

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Dataset Card

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
from datetime import date


@dataclass
class ColumnDoc:
    name: str
    dtype: str
    description: str
    nullable: bool = False
    example: str = ""


@dataclass
class DatasetCard:
    name: str
    version: str
    description: str
    source_systems: list[str]
    update_cadence: str        # e.g., "nightly at 02:30 UTC"
    row_count_approx: int
    date_range: tuple[date, date]
    columns: list[ColumnDoc]
    known_limitations: list[str]
    not_suitable_for: list[str]
    owner_team: str
    owner_contact: str
    pii_present: bool = False
    access_tier: str = "internal"  # internal | confidential | restricted

    def validate(self) -> list[str]:
        """Return list of issues. Empty = valid."""
        issues = []
        if not self.known_limitations:
            issues.append("known_limitations is empty — add at least one.")
        if not self.not_suitable_for:
            issues.append("not_suitable_for is empty — add at least one.")
        if not self.owner_contact:
            issues.append("owner_contact is missing.")
        if self.pii_present and self.access_tier == "internal":
            issues.append("PII dataset should not be access_tier='internal'.")
        return issues

    def render(self) -> str:
        lines = [
            f"# Dataset Card: {self.name} v{self.version}",
            f"\n{self.description}",
            f"\n**Source systems:** {', '.join(self.source_systems)}",
            f"**Update cadence:** {self.update_cadence}",
            f"**Approximate row count:** {self.row_count_approx:,}",
            f"**Date range:** {self.date_range[0]} → {self.date_range[1]}",
            f"**Access tier:** {self.access_tier} | **PII:** {self.pii_present}",
            f"**Owner:** {self.owner_team} ({self.owner_contact})",
            "\n## Schema",
        ]
        for col in self.columns:
            null_flag = " (nullable)" if col.nullable else ""
            lines.append(f"- `{col.name}` [{col.dtype}]{null_flag}: {col.description}"
                         + (f" · e.g., `{col.example}`" if col.example else ""))
        lines.append("\n## Known Limitations")
        for lim in self.known_limitations:
            lines.append(f"- {lim}")
        lines.append("\n## Not Suitable For")
        for ns in self.not_suitable_for:
            lines.append(f"- {ns}")
        return "\n".join(lines)


# --- Demo ---
card = DatasetCard(
    name="watch_events",
    version="1.2.0",
    description="Raw viewing events from the CinemaStream player. "
                "One row per play/pause/stop event per user per movie.",
    source_systems=["player_backend_api", "kinesis_stream_watch_events"],
    update_cadence="nightly at 02:30 UTC via Airflow watch_events_ingestion",
    row_count_approx=14_000_000,
    date_range=(date(2022, 1, 1), date(2026, 6, 15)),
    columns=[
        ColumnDoc("event_id",       "UUID",      "Unique event identifier"),
        ColumnDoc("user_id",        "INTEGER",   "Foreign key to users table"),
        ColumnDoc("movie_id",       "INTEGER",   "Foreign key to movies table"),
        ColumnDoc("event_type",     "VARCHAR",   "PLAY | PAUSE | STOP | SEEK",
                  example="PLAY"),
        ColumnDoc("watch_minutes",  "FLOAT",     "Minutes watched in this session",
                  nullable=True,
                  example="23.5"),
        ColumnDoc("completed",      "BOOLEAN",   "True if watch_minutes / duration >= 0.90",
                  nullable=True),
        ColumnDoc("device",         "VARCHAR",   "mobile | desktop | tv | tablet",
                  example="mobile"),
        ColumnDoc("country",        "VARCHAR",   "ISO 3166-1 alpha-2 country code",
                  example="SG"),
        ColumnDoc("event_ts",       "TIMESTAMP", "UTC timestamp of the event"),
    ],
    known_limitations=[
        "The `completed` flag was added 2024-03-01; rows before this date are NULL.",
        "SEEK events do not update `watch_minutes` — treat watch_minutes as a lower bound.",
        "country codes are user-reported at signup, not IP-derived; may differ from real location.",
        "Rows with watch_minutes < 0.5 are likely accidental plays (button mis-taps) — filter for serious analysis.",
    ],
    not_suitable_for=[
        "Precise engagement metrics before 2024-03-01 (completed is NULL).",
        "Real-time dashboards (batch pipeline, not streaming; up to 24h lag).",
        "Training models that require completed=True/False without a date filter.",
    ],
    owner_team="Data Engineering",
    owner_contact="priya@cinemastream.io",
    pii_present=False,
    access_tier="internal",
)

issues = card.validate()
if issues:
    print("INVALID:", issues)
else:
    print("Card valid.")
    print(card.render())

### 2.2 Model Card

In [ ]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class EvalSlice:
    name: str
    metric: str
    value: float
    note: str = ""


@dataclass
class ModelCard:
    model_id: str
    version: str
    model_type: str           # e.g., "RandomForestClassifier"
    task: str                 # e.g., "Binary classification"
    intended_use: str
    out_of_scope_use: list[str]
    training_data: str        # reference to a DatasetCard name + version
    training_date: str
    features: list[str]
    threshold: Optional[float]
    overall_metrics: dict[str, float]
    eval_slices: list[EvalSlice]
    known_biases: list[str]
    owner_team: str
    owner_contact: str
    artifact_path: str
    next_review_date: str

    def validate(self) -> list[str]:
        issues = []
        if not self.out_of_scope_use:
            issues.append("out_of_scope_use is empty.")
        if not self.eval_slices:
            issues.append("eval_slices is empty — add at least one breakdown.")
        if not self.known_biases:
            issues.append("known_biases is empty — add at least one.")
        if self.threshold is None:
            issues.append("threshold is None — specify the decision threshold.")
        return issues

    def render(self) -> str:
        lines = [
            f"# Model Card: {self.model_id} v{self.version}",
            f"\n**Type:** {self.model_type} | **Task:** {self.task}",
            f"\n## Intended Use\n{self.intended_use}",
            "\n## Out-of-Scope Use",
        ]
        for oos in self.out_of_scope_use:
            lines.append(f"- {oos}")
        lines.append(f"\n## Training Data\n{self.training_data}")
        lines.append(f"\n**Training date:** {self.training_date}")
        lines.append(f"\n**Features ({len(self.features)}):** "
                     + ", ".join(f"`{f}`" for f in self.features[:5])
                     + (f" + {len(self.features)-5} more" if len(self.features) > 5 else ""))
        lines.append(f"\n**Decision threshold:** {self.threshold}")
        lines.append("\n## Overall Metrics")
        for k, v in self.overall_metrics.items():
            lines.append(f"- {k}: {v:.3f}")
        lines.append("\n## Performance by Slice")
        for sl in self.eval_slices:
            note = f" ({sl.note})" if sl.note else ""
            lines.append(f"- {sl.name}: {sl.metric}={sl.value:.3f}{note}")
        lines.append("\n## Known Biases")
        for b in self.known_biases:
            lines.append(f"- {b}")
        lines.append(f"\n**Owner:** {self.owner_team} ({self.owner_contact})")
        lines.append(f"**Artifact:** `{self.artifact_path}`")
        lines.append(f"**Next review:** {self.next_review_date}")
        return "\n".join(lines)


churn_card = ModelCard(
    model_id="cinemastream-churn-v1",
    version="1.0.0",
    model_type="RandomForestClassifier (sklearn 1.x)",
    task="Binary classification — predict subscriber churn within 30 days",
    intended_use=(
        "Flag at-risk subscribers for CinemaStream's retention team. "
        "Outputs a probability score and a binary flag at threshold=0.3. "
        "Designed for weekly batch scoring of active subscribers only."
    ),
    out_of_scope_use=[
        "Real-time churn prediction (batch-only; scoring freshness is weekly).",
        "Predicting churn for non-subscriber users (Free plan with no subscription history).",
        "Use by external parties or partner companies without a signed data agreement.",
        "Making automated cancellation or billing decisions without human review.",
    ],
    training_data="watch_events v1.2.0 + subscriptions v1.0 + synthetic churn labels "
                  "(seed=42, n=300). Labels generated from canonical per-country/plan churn "
                  "rates (Ch067). Training cutoff: 2026-05-01.",
    training_date="2026-05-15",
    features=[
        "watch_minutes_avg", "days_since_last_watch", "tenure_months",
        "support_tickets_count", "engagement_score", "country_TH",
        "country_VN", "tenure_bucket_established",
    ],
    threshold=0.3,
    overall_metrics={
        "ROC-AUC":   0.829,
        "Recall":    0.600,
        "Precision": 0.214,
        "F1":        0.316,
        "Accuracy":  0.827,
    },
    eval_slices=[
        EvalSlice("VN subscribers",      "Recall", 0.70,
                  "Highest churn rate in training data (8.2%); model catches ~70%"),
        EvalSlice("PH subscribers",      "Recall", 0.60,
                  "Moderate churn (6.5%); aligns with overall recall"),
        EvalSlice("SG Premium plan",     "Recall", 0.30,
                  "Low churn (2.5%); model misses most — small n in training"),
        EvalSlice("< 3 months tenure",   "Precision", 0.11,
                  "High false-positive rate on new users — noisy behaviour"),
        EvalSlice("No watch events 30d", "Recall",  0.82,
                  "Strongest signal: inactive users correctly flagged"),
    ],
    known_biases=[
        "Training dataset is synthetic (n=300); real-world performance may differ at scale.",
        "Under-represents SG and MY Premium subscribers — insufficient positive examples in training.",
        "Users with tenure < 3 months are systematically over-flagged (low precision for this slice).",
        "Churn labels generated from observed aggregate rates, not ground-truth future churn events.",
    ],
    owner_team="ML Engineering",
    owner_contact="mei@cinemastream.io",
    artifact_path="cinemastream/ml/churn/artifacts/champion.joblib",
    next_review_date="2026-09-15",
)

issues = churn_card.validate()
print("Issues:", issues if issues else "none")
print(churn_card.render())

### 2.3 Validation as a gate

In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class IncompleteModelCard:
    model_id: str
    intended_use: str
    out_of_scope_use: list[str]
    eval_slices: list[str]
    known_biases: list[str]
    threshold: Optional[float]

    def validate(self) -> list[str]:
        issues = []
        if not self.out_of_scope_use:
            issues.append("out_of_scope_use is empty.")
        if not self.eval_slices:
            issues.append("eval_slices is empty — add at least one breakdown.")
        if not self.known_biases:
            issues.append("known_biases is empty — add at least one.")
        if self.threshold is None:
            issues.append("threshold is None — specify the decision threshold.")
        return issues


def promote_to_production(card: IncompleteModelCard) -> None:
    issues = card.validate()
    if issues:
        raise ValueError(f"Model card incomplete — cannot promote:\n" +
                         "\n".join(f"  • {i}" for i in issues))
    print(f"Model {card.model_id} promoted to production.")


# This will fail
bad_card = IncompleteModelCard(
    model_id="my-new-model",
    intended_use="Predict something important.",
    out_of_scope_use=[],     # forgot this
    eval_slices=[],          # forgot this
    known_biases=[],         # forgot this
    threshold=None,          # forgot this
)

try:
    promote_to_production(bad_card)
except ValueError as e:
    print(e)

## 3. CinemaStream in Practice

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
from datetime import date


# --- Minimal production-ready versions of the card classes ---

@dataclass
class DatasetCard:
    name: str
    version: str
    description: str
    update_cadence: str
    known_limitations: list[str]
    not_suitable_for: list[str]
    owner_contact: str
    pii_present: bool = False

    def validate(self) -> list[str]:
        issues = []
        if not self.known_limitations:
            issues.append("known_limitations empty.")
        if not self.not_suitable_for:
            issues.append("not_suitable_for empty.")
        if not self.owner_contact:
            issues.append("owner_contact missing.")
        return issues


@dataclass
class EvalSlice:
    name: str
    metric: str
    value: float
    note: str = ""


@dataclass
class ModelCard:
    model_id: str
    version: str
    intended_use: str
    out_of_scope_use: list[str]
    training_data_ref: str
    threshold: Optional[float]
    overall_metrics: dict[str, float]
    eval_slices: list[EvalSlice]
    known_biases: list[str]
    owner_contact: str
    next_review_date: str

    def validate(self) -> list[str]:
        issues = []
        if not self.out_of_scope_use:
            issues.append("out_of_scope_use empty.")
        if not self.eval_slices:
            issues.append("eval_slices empty.")
        if not self.known_biases:
            issues.append("known_biases empty.")
        if self.threshold is None:
            issues.append("threshold missing.")
        return issues


# --- Priya writes the watch_events dataset card ---
watch_events_card = DatasetCard(
    name="watch_events",
    version="1.2.0",
    description="Raw CinemaStream player events. One row per play/pause/stop/seek per user per movie.",
    update_cadence="Nightly 02:30 UTC via Airflow watch_events_ingestion.",
    known_limitations=[
        "`completed` flag NULL before 2024-03-01.",
        "SEEK events do not update watch_minutes — use as lower bound.",
        "country is user-reported at signup, not IP-derived.",
    ],
    not_suitable_for=[
        "Completion metrics before 2024-03-01.",
        "Real-time dashboards (up to 24h lag).",
        "Training churn models without filtering completed=True/False by date.",
    ],
    owner_contact="priya@cinemastream.io",
    pii_present=False,
)

# --- Mei writes the churn model card ---
churn_model_card = ModelCard(
    model_id="cinemastream-churn-v1",
    version="1.0.0",
    intended_use=(
        "Weekly batch scoring of active subscribers to flag at-risk users "
        "for CinemaStream's retention team. Binary flag at threshold=0.3."
    ),
    out_of_scope_use=[
        "Real-time scoring (batch only; weekly cadence).",
        "Free-plan users with no subscription history.",
        "Automated billing decisions without human review.",
        "External or partner company use without signed data agreement.",
    ],
    training_data_ref="watch_events v1.2.0 + subscriptions v1.0 (synthetic, n=300, seed=42)",
    threshold=0.3,
    overall_metrics={
        "ROC-AUC": 0.829, "Recall": 0.600,
        "Precision": 0.214, "F1": 0.316,
    },
    eval_slices=[
        EvalSlice("VN subscribers",      "Recall",    0.70,
                  "Highest-churn cohort; model catches ~70%"),
        EvalSlice("SG Premium",          "Recall",    0.30,
                  "Lowest churn (2.5%); under-represented in training"),
        EvalSlice("Tenure < 3 months",   "Precision", 0.11,
                  "Over-flagged; noisy early-lifecycle behaviour"),
        EvalSlice("No activity in 30d",  "Recall",    0.82,
                  "Strongest signal; inactive users reliably caught"),
    ],
    known_biases=[
        "Trained on synthetic labels (n=300); real performance may differ at scale.",
        "SG/MY Premium cohort under-represented — low recall for that slice.",
        "New-subscriber over-flagging (Precision=0.11 for tenure < 3 months).",
    ],
    owner_contact="mei@cinemastream.io",
    next_review_date="2026-09-15",
)

# --- Run validation ---
for card in [watch_events_card, churn_model_card]:
    issues = card.validate()
    name = getattr(card, 'name', None) or getattr(card, 'model_id', '')
    status = "✓ VALID" if not issues else f"✗ INVALID: {issues}"
    print(f"{name}: {status}")

# --- Dharani's question answered by reading the model card ---
print("\nDharani asks: 'Can we trigger discount codes automatically?'")
oos = churn_model_card.out_of_scope_use
automated = [u for u in oos if "automated" in u.lower() or "billing" in u.lower()]
print(f"Out-of-scope use covers this: '{automated[0]}'")
print("Answer: No. Human review required before any billing action.")

# --- Rohan's question answered by reading the eval slices ---
print("\nRohan asks: 'How does the model perform on SG Premium users?'")
sg_slice = next(s for s in churn_model_card.eval_slices if "SG" in s.name)
print(f"Slice '{sg_slice.name}': {sg_slice.metric}={sg_slice.value:.2f}. "
      f"Note: {sg_slice.note}")
print("Answer: Recall=0.30 — model misses ~70% of SG Premium churners. "
      "Don't rely on it for that cohort without retraining on more SG data.")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class DatasetCard:
    name: str
    version: str
    description: str
    update_cadence: str
    known_limitations: list[str]
    not_suitable_for: list[str]
    owner_contact: str
    pii_present: bool = False
    access_tier: str = "internal"

    def validate(self) -> list[str]:
        issues = []
        if not self.known_limitations:
            issues.append("known_limitations empty.")
        if not self.not_suitable_for:
            issues.append("not_suitable_for empty.")
        if not self.owner_contact:
            issues.append("owner_contact missing.")
        if self.pii_present and self.access_tier == "internal":
            issues.append("PII dataset should not be access_tier='internal'.")
        return issues


support_card = DatasetCard(
    name="support_tickets",
    version="1.0.0",
    description="Weekly Zendesk export of customer support tickets.",
    update_cadence="Weekly CSV export every Monday 09:00 UTC.",
    known_limitations=[
        "`category` column was recoded 2024-01-01; pre-2024 values use legacy taxonomy "
        "and do not map 1:1 to post-2024 categories.",
    ],
    not_suitable_for=[
        "Trend analysis spanning the 2023–2024 recoding boundary without a category mapping.",
        "Direct user identification (user_id links to PII in users table — join only via approved query).",
    ],
    owner_contact="support-data@company.io",
    pii_present=True,
    access_tier="confidential",
)

issues = support_card.validate()
print("Issues:", issues if issues else "none")
print(f"Card for '{support_card.name}' is {'valid' if not issues else 'invalid'}.")

In [ ]:
from dataclasses import dataclass


@dataclass
class DatasetCard:
    name: str
    version: str
    description: str
    update_cadence: str
    known_limitations: list[str]
    not_suitable_for: list[str]
    owner_contact: str
    pii_present: bool = False

    def validate(self) -> list[str]:
        issues = []
        if not self.known_limitations:
            issues.append("known_limitations empty.")
        if not self.not_suitable_for:
            issues.append("not_suitable_for empty.")
        if not self.owner_contact:
            issues.append("owner_contact missing.")
        return issues


filmibox_wiki_card = DatasetCard(
    name="filmibox_wiki",
    version="1.0.0",
    description=(
        "87 Confluence articles from FilmiBox internal wiki, "
        "bilingual (English + Hindi), HTML-stripped to plain text."
    ),
    update_cadence="Weekly export every Sunday 23:00 IST.",
    known_limitations=[
        "Articles are not language-tagged — English/Hindi detection must be done programmatically.",
        "Content accuracy not guaranteed — wiki is community-edited and may contain errors.",
    ],
    not_suitable_for=[
        "Regulatory, legal, or compliance queries (content is not legally reviewed).",
    ],
    owner_contact="dev@filmibox.io",
    pii_present=False,
)

issues = filmibox_wiki_card.validate()
print("Issues:", issues if issues else "none")
print(f"Card: {filmibox_wiki_card.name} v{filmibox_wiki_card.version}")